# 0. Introduction

<pre style="line-height:1.35">
  competition train.csv (1,460)        fold table (Id, fold)        competition test.csv (1,459)
            |                                   |                              |
            +-- fix contradictions (-7) --------+                              |
            +-- fill missing -------------------+                              |
            |                                                                  |
            +-- encode 43 categoricals ---> 23 ordinal | 21 derived | 11 OOF-TE +
            |                                                                  |
            +-- 5-fold split (1,453) ------------------------------------------+
                        |
                        v   per fold
            design matrix : 1,162 x 90 terminals
                        |
                        v
        +---------------------------------------------------+
        |  GP   population 10,000                           |
        |   evaluate : genes -> matrix -> ridge (GCV picks a)|
        |   rank     : GCV + parsimony penalty               |
        |   breed    : elite 1 + tournament 15               |
        +---------------------------------------------------+
                        |  x 300 generations
                        v
            returned individual  (GPTIPS rule : valid picks the snapshot)
                        |
                        v
            5 fold models --> mean in log space --> submission.csv
</pre>

Figure 1 : the pipeline. The GP writes 1–15 columns out of 90 terminals, the ridge turns them into a price.

### **Summary**
A ridge can only shrink the columns it is handed. It will never write `GrLivArea × unit_price`, and it cannot
delete a dead column either. So the two jobs go to two solvers: **the GP fits the shape of each column, the
ridge divides the weight among them.** An individual carries 1–15 expression trees over 90 terminals and 8
protected operators; the ridge reads those columns, picks its own α by GCV, and that GCV is the fitness. What
makes it affordable is the ridge — one solve over a 6-point α grid costs 0.37 ms, so a population of 10,000 can
be scored 300 generations over. The submission is the mean of five fold models in log space.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">Solver</th><th style="text-align:left">Decides</th><th style="text-align:center">Cost / eval</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>Multi-gene GP</b></td><td style="text-align:left">which columns exist, and how many</td><td style="text-align:center">—</td></tr>
<tr><td style="text-align:left"><b>Ridge + GCV</b></td><td style="text-align:left">the weight of each column, and its own α</td><td style="text-align:center">0.37 ms</td></tr>
</tbody></table>
</div><br>

<hr>

### **GP가 식의 모양을 맞추고, 릿지가 비중을 나눈다**
릿지는 받은 열을 줄이는 일만 한다. `GrLivArea × 평당가` 같은 열을 새로 쓰지 못하고, 쓸모없는 열을 지우지도
못한다. 계수를 0 쪽으로 미는 것이 전부다.<br>
<br>
그래서 두 일을 두 풀이에 나눠 준다. 유전 프로그래밍은 수식을 탐색한다 — 열의 모양을 어떻게 할 것인가.
릿지는 그 열을 받아 비중을 나눈다 — 각 열이 얼마나 셀 것인가. 서로의 일을 하지 않는다. 적합도는 릿지의
GCV이므로 GP는 가격을 직접 보지 않는다. '릿지가 좋아하는 재료'만 배운다.<br>
<br>
개체 하나는 터미널 90개와 보호 연산자 8개로 만든 수식 나무를 **1~15개** 들고 있다. 릿지는 그 열들을 읽고
**개체마다 GCV가 최소인 α**를 골라 가격을 내놓는다.<br>
<br>
이 분업이 감당되는 이유는 릿지가 싸기 때문이다. α 격자 6개를 다 풀어도 0.37ms라서 개체 10,000개를
300세대에 걸쳐 채점할 수 있다. 그 자리에 부스터를 앉히면 평가 단가가 227배가 되고, 같은 예산에서는
릿지가 더 나았다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">누가</th><th style="text-align:left">무엇을 정하는가</th><th style="text-align:center">평가 1회 단가</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>다유전자 GP</b></td><td style="text-align:left">어떤 열이 몇 개 존재할지</td><td style="text-align:center">—</td></tr>
<tr><td style="text-align:left"><b>릿지 + GCV</b></td><td style="text-align:left">각 열의 비중, 그리고 자기 α</td><td style="text-align:center">0.37 ms</td></tr>
</tbody></table>
</div><br>
제출물은 fold 모델 5개의 예측을 log 공간에서 평균한 것이다.

### **왜 선형 결합은 GP에게 주지 않는가**
처음에는 선형 결합까지 GP에게 전부 맡겨 봤다. 세 군데서 어긋났다.<br>
<br>
- **학습 데이터조차 못 맞췄다.** 같은 열을 릿지에 그냥 주면 valid가 0.12인데 GP는 train이 0.14에 머물렀다.
세대를 세 배로 늘려도 거의 내려가지 않았다.<br>
- **상수가 죽었다.** log 가격은 절편 없이 맞출 수 없는데, 상수를 가진 개체는 0세대에도 드물었고 곧 사라졌다.
개체군은 대신 `KitchenAbvGr`처럼 거의 모든 행이 같은 값인 열을 상수처럼 썼다. 최종 개체군의 91%에 그
수법이 들어 있었다.<br>
- **범주형 원핫이 지워졌다.** 105열 중 쓰인 것은 열 개 남짓이었다. Kronberger 외(GECCO 2018)가 같은 경향을
보고한다. GP는 명목형 지시 변수를 제거한다.<br>
<br>
릿지는 절편이 있고 모든 열에 계수가 하나씩 있다. 선형 결합은 거기로 넘긴다. 범주형은 GP가 보기 전에 모양을
바꿔 둔다. 글자 열 43개를 두 갈래로 가른다.<br>
<br>
1. **순서가 있는 것** → 정수 순서. `ExterQual`·`ExterCond`·`BsmtQual`·`BsmtCond`·`HeatingQC`·`KitchenQual`·
`FireplaceQu`·`GarageQual`·`GarageCond`·`PoolQC`가 `Ex`=5 … `Po`=1이고,
`BsmtExposure`·`BsmtFinType1/2`·`Functional`·`GarageFinish`·`LandSlope`·`LotShape`·`PavedDrive`·`Alley`·
`Electrical`·`Utilities`에도 자연스러운 순서가 있다. `CentralAir`·`Street`는 0/1이다. 1열이고 추정 잡음도
누출도 없다. 타깃 인코딩보다 낫다.<br>
   `Neighborhood`·`Exterior1st/2nd`·`Condition1/2`·`BldgType`·`HouseStyle`·`LotConfig`·`Fence`는 수준이 많아
   도메인 묶음으로 파생 열 21개를 만든다. 동네는 평당가 한 열로 들어간다.<br>
2. **순서가 없는 11개** → 아웃오브폴드 타깃 인코딩.
`MSZoning`·`LandContour`·`RoofStyle`·`RoofMatl`·`Foundation`·`Heating`·`MiscFeature`·`SaleType`·
`SaleCondition`·`GarageType`·`MasVnrType`. 사람이 순서를 정할 수 없으니 정답이 순서를 공급해야 한다.
원핫으로 주면 위에서 본 대로 GP가 지운다.<br>
<br>
LOO가 아니라 아웃오브폴드를 쓰는 이유는 3절에 수식으로 적었다.

### **유전 프로그래밍이 도는 방식**
한 세대는 평가 → 순위 → 번식의 한 바퀴다.<br>
<br>
0. **초기화, 0세대만.** 개체마다 유전자 개수를 1~15에서 균등하게 뽑고, 각 나무를 높이 2~6에서 무작위로 심는다.
그래서 출발 개체군의 평균은 8개 안팎이다. 이 수는 고정이 아니라 진화한다. 세대가 가면서 상한 쪽으로 올라간다.<br>
1. **평가.** 유전자를 행렬로 만들고 릿지를 푼다. 적합도 = GCV + 크기에 대한 절약 벌점.<br>
2. **순위.** 적합도가 낮은 쪽이 이긴다. 같으면 짧은 프로그램이 이긴다.<br>
3. **번식.** 1등은 손대지 않고 그대로 넘긴다. 남은 9,999자리는 토너먼트로 채운다. 무작위로 15개를 뽑아
그중 최고를 부모로 삼고 반복한다.<br>
<br>
뽑힌 부모 한 쌍은 연산자 하나를 정확히 한 번 거친다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">연산자</th><th style="text-align:center">비율</th><th style="text-align:left">무엇이 바뀌는가</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>상위 교차</b></td><td style="text-align:center">14%</td><td style="text-align:left">유전자를 통째로 주고받는다. 유전자 개수가 바뀐다</td></tr>
<tr><td style="text-align:left"><b>하위 교차</b></td><td style="text-align:center">56%</td><td style="text-align:left">유전자 하나의 가지를 맞바꾼다</td></tr>
<tr><td style="text-align:left"><b>가지 돌연변이</b></td><td style="text-align:center">24%</td><td style="text-align:left">가지 하나를 새 무작위 가지로 바꾼다</td></tr>
<tr><td style="text-align:left"><b>유전자 획득 / 상실</b></td><td style="text-align:center">3% / 3%</td><td style="text-align:left">유전자를 하나 붙이거나 뗀다</td></tr>
</tbody></table>
</div><br>
높이 6을 넘는 결과는 버리고 부모를 그대로 둔다. 한 개체 안에서 같은 식이 중복되면 하나로 줄인다.<br>
<br>
선택압은 전부 토너먼트 크기 15에서 나온다. 개체군 10,000 기준이다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">계층</th><th style="text-align:center">개체 수</th><th style="text-align:center">전체 부모 중 지분</th><th style="text-align:center">개체당 기대 자손</th></tr></thead>
<tbody>
<tr><td style="text-align:left">1등</td><td style="text-align:center">1</td><td style="text-align:center">0.15%</td><td style="text-align:center">14.99</td></tr>
<tr><td style="text-align:left">상위 1%</td><td style="text-align:center">100</td><td style="text-align:center">13.99%</td><td style="text-align:center">13.99</td></tr>
<tr><td style="text-align:left">상위 5%</td><td style="text-align:center">500</td><td style="text-align:center">53.67%</td><td style="text-align:center">10.73</td></tr>
<tr><td style="text-align:left">상위 10%</td><td style="text-align:center">1,000</td><td style="text-align:center">79.41%</td><td style="text-align:center">7.94</td></tr>
<tr><td style="text-align:left"><b>상위 20%</b></td><td style="text-align:center">2,000</td><td style="text-align:center"><b>96.48%</b></td><td style="text-align:center">4.82</td></tr>
<tr><td style="text-align:left">하위 50%</td><td style="text-align:center">5,000</td><td style="text-align:center">0.003%</td><td style="text-align:center">0.00</td></tr>
</tbody></table>
</div><br>
상위 20%가 다음 세대 유전자의 96.48%를 남긴다. 1 − 0.8¹⁵이다. 자손을 하나라도 남기는 개체는 20.6%뿐이고
하위 절반은 사실상 0이다. 0.5¹⁵ = 0.003%다.<br>
<br>
절단 선택으로 상위 20%만 부모로 쓰면 선택 강도는 같고 다양성만 잃는다. 토너먼트는 패자를 추첨에 남겨 둔다
(Blickle &amp; Thiele, *Evolutionary Computation* 4(4), 1996).

# 1. Import

In [1]:
# Thread caps must be set BEFORE numpy is imported, or BLAS will already have spawned its pool.
# BLAS 스레드 상한은 **numpy import 전에** 걸어야 한다. 늦으면 이미 스레드 풀이 만들어져 있다.
import os

for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_v] = '1'

import json, operator, random, resource, shutil, time, zipfile
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass
from pathlib import Path

# Turn off all warnings
# 경고 off
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from deap import creator, base, gp, tools

print('numpy  :', np.__version__)
print('pandas :', pd.__version__)
print('deap   : ok')

numpy  : 2.1.3
pandas : 2.3.3
deap   : ok


### Environment check / 환경 점검

In [2]:
# This pipeline is pure numpy on the CPU — the GPU of a T4 x 2 session is not used.
# 이 파이프라인은 CPU numpy만 쓴다. T4 x 2 세션의 GPU는 쓰지 않는다(쓸 자리가 없다).
N_CPU = os.cpu_count() or 1
RSS_MB = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

print('cpu count       :', N_CPU)
print('thread cap      :', os.environ['OMP_NUM_THREADS'], '(per process)')
print(f'max RSS so far  : {RSS_MB():.0f} MB')
try:
    _mem = int(next(l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemAvailable'))) / 1e6
    print(f'memory available: {_mem:.1f} GB')
except Exception:
    pass

# One worker per core. Five folds on four cores run as two waves — that is fine and keeps RSS flat.
# 코어 하나에 워커 하나. fold 5개를 코어 4개로 돌리면 두 물결이 되는데, 그게 메모리를 가장 평평하게 유지한다.
PROCS = max(1, min(5, N_CPU))
print('worker processes:', PROCS)

cpu count       : 4
thread cap      : 1 (per process)
max RSS so far  : 232 MB
memory available: 31.6 GB
worker processes: 4


### Directory setup

In [3]:
# 경로 설정
if Path('/kaggle/input').exists():
    # when Kaggle
    ROOT_DIR = Path('/kaggle/working')
    COMP_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('train.csv'))
                    if (p.parent / 'test.csv').exists())      # 대회 데이터 / competition data
    SPLIT_DIR = next(p.parent for p in sorted(Path('/kaggle/input').rglob('folds.csv')))
    print('Environment : Kaggle')
else:
    # when local
    ROOT_DIR = Path(os.environ.get('HP_KAGGLE_WORK', '.'))
    COMP_DIR = Path(os.environ.get('HP_COMP_DIR', './origin'))
    SPLIT_DIR = Path(os.environ.get('HP_SPLIT_DIR', './export/kaggle/ids'))
    print('Environment : local')

OUTPUT_DIR = ROOT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUB_PATH = ROOT_DIR / 'submission.csv'

print(f'ROOT_DIR   = {ROOT_DIR}')
print(f'COMP_DIR   = {COMP_DIR}')
print(f'SPLIT_DIR  = {SPLIT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

Environment : Kaggle
ROOT_DIR   = /kaggle/working
COMP_DIR   = /kaggle/input/competitions/house-prices-advanced-regression-techniques
SPLIT_DIR  = /kaggle/input/datasets/illuminatep/hp-bundle
OUTPUT_DIR = /kaggle/working/outputs


### Check assets

In [4]:
# 자산 검사
FOLDS_PATH = SPLIT_DIR / 'folds.csv'

_bad = []
for _f in ('train.csv', 'test.csv', 'sample_submission.csv'):
    if not (COMP_DIR / _f).exists():
        _bad.append(f'Missing {_f} ({COMP_DIR}) — attach the competition data')
if not FOLDS_PATH.exists():
    _bad.append(f'Missing folds.csv ({SPLIT_DIR}) — attach the split dataset')

assert not _bad, 'Asset check failed!!:\n  ' + '\n  '.join(_bad)
print('*** Asset check passed ***')

*** Asset check passed ***


### Flag : True for a quick wiring run, False for the full budget

In [5]:
# Quick run = True (a few minutes, proves the wiring) / Full budget = False
# 배선 확인 = True(몇 분) · 본 런 = False(fold당 약 20분)
QUICK = False
MAX_HOURS = 7.0        # A Kaggle session is 9 h — stop with room to spare / 세션 9시간이라 여유를 둔다
print(f'Mode: {"QUICK" if QUICK else "FULL"}')

Mode: FULL


### Set seed and budget / 시드와 예산

In [6]:
# It must be 42, since it is the ultimate answer to life, the universe and everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed = {SEED}')

CFG = dict(encoding='oof',
           population=200 if QUICK else 10000,
           max_gen=3 if QUICK else 300,
           genes=15, max_height=6, parsimony=1e-4, alpha_theta=0.0,
           lm_early=0, lm_early_gens=0, lm_late=0,          # LM off — theta refinement is not used here / theta 정련은 쓰지 않는다
           parsimony_decay='sqrt_gen', parsimony_scale=1.0, parsimony_target=None,
           size_measure='linear', covariant_parsimony=True,
           # fold 5개을 워커 PROCS개로 돌리면 물결은 올림(5/PROCS)이다. 내림으로 나누면 fold마다
           # 예산을 통째로 줘서 총 벽시계가 세션 한도를 넘는다 (PROCS=4면 7h가 아니라 14h).
           # Waves is ceil(5/PROCS), not floor — flooring hands each fold the whole budget.
           seed=SEED, max_seconds=MAX_HOURS * 3600 / max(1, -(-5 // PROCS)),
           # Per-generation records for the figures / 그림을 그리기 위한 세대별 기록
           ckpt_top=5,        # 세대마다 남길 상위 개체 수 (1등 계보와 세대별 1등 표의 재료)
           census_top=50,     # 세대마다 남길 '가장 흔한 유전자식' 개수
           lineage=1)         # 개체군 전부의 부모-자식 간선 (0이면 끈다)

print(f'pop {CFG["population"]} x {CFG["max_gen"]} generations · genes 1..{CFG["genes"]} (variable)')
print(f'wall-clock cap per fold: {CFG["max_seconds"] / 3600:.1f} h')
print(f'records per generation: top {CFG["ckpt_top"]} individuals · census top {CFG["census_top"]}'
      f' · lineage {"on" if CFG["lineage"] else "off"}')

Seed = 42
pop 10000 x 300 generations · genes 1..15 (variable)
wall-clock cap per fold: 3.5 h
records per generation: top 5 individuals · census top 50 · lineage on


# 2. Load the split dataset / 데이터셋 불러오기

붙인 데이터셋은 `Id, fold` fold 표 하나다. 행은 대회 `train.csv`에서 Id로 다시 골라낸다.<br>
<br>
학습은 쓸 수 있는 전부다. 모순 7행을 지우면 1,453행이고, 따로 떼어 두는 행은 없다. 점수는 리더보드가 낸다.<br>
<br>
fold의 검증 행(fold마다 290~291행)은 그대로 남는다. GPTIPS 규칙이 반환할 개체를 고르는 데 쓴다.

In [7]:
# The fold table is the only thing shared — the rows come from the competition csv.
# 공유물은 fold 표뿐이다. 행은 대회 csv에서 가져온다.
NA_OPTS_LOAD = dict(keep_default_na=False, na_values=['NA'])    # 'NA'만 결측으로 / only 'NA' is missing

train_csv = pd.read_csv(COMP_DIR / 'train.csv', **NA_OPTS_LOAD)
test_csv = pd.read_csv(COMP_DIR / 'test.csv', **NA_OPTS_LOAD)
fold_tbl = pd.read_csv(FOLDS_PATH)

assert list(fold_tbl.columns) == ['Id', 'fold'], f'fold table columns {list(fold_tbl.columns)}!!'
assert fold_tbl.Id.is_unique, 'duplicated Id in the fold table!!'
assert fold_tbl.Id.isin(train_csv.Id).all(), \
    f'{int((~fold_tbl.Id.isin(train_csv.Id)).sum())} fold ids are not in the competition train.csv!!'

FOLD_OF = dict(zip(fold_tbl.Id, fold_tbl.fold))
print(f'train.csv {len(train_csv):,} rows · test.csv {len(test_csv):,} rows')
print(f'fold table {len(fold_tbl):,} rows · sizes {np.bincount(fold_tbl.fold).tolist()}')

train.csv 1,460 rows · test.csv 1,459 rows
fold table 1,453 rows · sizes [291, 291, 291, 290, 290]


### Id sum check / Id 합 검증

In [8]:
# The fold table must be the competition train.csv minus the rows that contradict themselves.
# fold 표는 대회 train.csv에서 자기 모순인 행만 뺀 것이어야 한다.
_extra = sorted(int(i) for i in set(train_csv.Id) - set(fold_tbl.Id))

print(f'{"item":<12}{"rows":>8}{"id sum":>14}')
for _k, _df in (('train.csv', train_csv), ('test.csv', test_csv), ('fold table', fold_tbl)):
    print(f'{_k:<12}{len(_df):>8,}{int(_df.Id.sum()):>14,}')

assert set(fold_tbl.Id) <= set(train_csv.Id), 'the fold table has ids that train.csv does not!!'
assert len(_extra) == 7, f'train.csv has {len(_extra)} rows outside the fold table, expected 7!!'
assert int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()) == sum(_extra), 'id sum does not add up!!'

print(f'\n*** Id check passed ***')
print(f'    fold table = train.csv minus the 7 contradictory rows {_extra}')
print(f'    id sum gap {int(train_csv.Id.sum()) - int(fold_tbl.Id.sum()):,} = sum of those 7 ids')

item            rows        id sum
train.csv      1,460     1,066,530
test.csv       1,459     3,195,210
fold table     1,453     1,059,333

*** Id check passed ***
    fold table = train.csv minus the 7 contradictory rows [625, 689, 774, 1231, 1242, 1301, 1335]
    id sum gap 7,197 = sum of those 7 ids


# 3. Preprocessing / 전처리

원시 csv가 수치형 터미널 90개가 되는 과정은 다섯 단계다.<br>
<br>
1. **모순 처리.** 학습 7행이 자기 자신과 어긋난다. 지하실 유형이 없는데 지하실 면적이 있는 식이다.
그 7행을 지우고 6개 칸은 값을 고친다.<br>
2. **결측 채우기.** `NA`가 어떤 열에서는 '없음'이고 어떤 열에서는 '모름'이다. 두 경우를 다르게 채운다.<br>
3. **범주형 인코딩.** 글자 열 43개, 세 갈래다.<br>
4. **누출 통제.** 정답을 건드리는 통계는 모두 그 fold의 학습 행으로만 적합한다.<br>
5. **조립.** 가공한 79열 + 아웃오브폴드 타깃 인코딩 11열.<br>

## Rows — contradictions and missing values / 행 — 모순과 결측

### Fix contradictions and fill missing / 모순 처리와 결측 채우기

In [9]:
# Stage 1-2 — remove self-contradicting rows, then fill the two kinds of NA.
# 1~2단계 — 자기 모순인 행을 지우고, 두 종류의 NA를 각각 다르게 채운다.
#   NA_IS_NONE   : 'NA'가 '없음'이라는 뜻인 열 (지하실 없음, 차고 없음 ...)
#   FIXED_FILL   : 'NA'가 '모름'이라서 정해진 값으로 채우는 열
#   PAIR         : 한쪽이 없으면 다른 쪽도 없어야 하는 짝 (지하실 유형 ↔ 면적)


# 'NA'만 결측으로 읽는다. MasVnrType의 'None'은 값이다.
NA_OPTS = dict(keep_default_na=False, na_values=['NA'])


# Seven training rows contradict themselves (a basement area with no basement type). They go out.
# 학습 7행이 자기 자신과 어긋난다(지하실 유형이 없는데 면적이 있는 식). 지운다.
TRAIN_DROP_IDS = (625, 689, 774, 1231, 1242, 1301, 1335)

TEST_YEAR_FIX_IDS = (1877, 2550, 2593)

TEST_MASVNR_FIX_IDS = (1670, 2453, 2320)

BSMT_CATS = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']

BSMT_NUMS = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']

GARAGE_CATS = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

GARAGE_NUMS = ['GarageYrBlt', 'GarageCars', 'GarageArea']

PAIR = {**{c: ('TotalBsmtSF', 'NA') for c in BSMT_CATS},
        **{c: ('GarageArea', 'NA') for c in GARAGE_CATS},
        'FireplaceQu': ('Fireplaces', 'NA'),
        'PoolQC': ('PoolArea', 'NA'),
        'MasVnrType': ('MasVnrArea', 'None')}


# 'NA' means "none" in these columns and "unknown" elsewhere. The two cases are filled differently.
# 이 열들에서 'NA'는 '없음'이고, 다른 열에서는 '모름'이다. 두 경우를 다르게 채운다.
NA_IS_NONE = ['Alley', 'Fence', 'MiscFeature']


# NA가 모름인 열. 괄호 안 값으로 채운다.
FIXED_FILL = {'MSZoning': 'RL', 'Utilities': 'AllPub', 'Exterior1st': 'VinylSd', 'Exterior2nd': 'VinylSd',
              'KitchenQual': 'TA', 'Functional': 'Typ', 'SaleType': 'WD', 'Electrical': 'SBrkr'}


# 면적이 0보다 큰데 유형이 NA인 행에 넣을 값.
MASVNR_TYPE_WITH_AREA = 'BrkFace'

def numeric_columns(df):
    return [c for c in df.columns if c not in ('Id', 'SalePrice') and pd.api.types.is_numeric_dtype(df[c])]


# Fix what can be fixed, drop what cannot. Test rows are never dropped — the submission needs all 1,459.
# 고칠 수 있으면 고치고 못 고치면 버린다. test 행은 절대 버리지 않는다. 제출에 1,459행이 다 필요하다.
def fix_contradictions(train, test):
    train = train[~train['Id'].isin(TRAIN_DROP_IDS)].reset_index(drop=True)
    test = test.copy()
    changes = []

    def change(mask, col, new, rule):
        for idx in test.index[mask]:
            value = new[idx] if isinstance(new, pd.Series) else new
            if test.at[idx, col] != value:
                changes.append(dict(Id=int(test.at[idx, 'Id']), column=col, before=test.at[idx, col], after=value,
                                    rule=rule))
            test.at[idx, col] = value

    change(test['GarageYrBlt'] == 2207, 'GarageYrBlt', 2007, 'GarageYrBlt 2207 → 2007')
    change(test['YearRemodAdd'] < test['YearBuilt'], 'YearRemodAdd', test['YearBuilt'], 'YearRemodAdd < YearBuilt → YearBuilt')
    change(test['YrSold'] < test['YearBuilt'], 'YrSold', test['YearBuilt'], 'YrSold < YearBuilt → YearBuilt')
    area_one = test['MasVnrArea'] == 1
    change(area_one, 'MasVnrType', 'None', '마감재 면적 1 → 0·None')
    change(area_one, 'MasVnrArea', 0.0, '마감재 면적 1 → 0·None')
    change((test['MasVnrType'] == 'None') & (test['MasVnrArea'] > 1), 'MasVnrType', 'BrkFace', 'None + 면적 > 1 → BrkFace')
    has_type = test['MasVnrType'].notna() & (test['MasVnrType'] != 'None')
    change(has_type & (test['MasVnrArea'] == 0), 'MasVnrType', 'None', '종류 있음 + 면적 0 → None')
    return train, test, pd.DataFrame(changes, columns=['Id', 'column', 'before', 'after', 'rule'])

def _mode(values):
    modes = values.mode()
    return modes.iloc[0], list(modes)


# Paired columns decide whether an NA is "none" or "unknown": if the partner is 0 it is none, else unknown.
# 짝 컬럼으로 NA의 뜻을 가린다. 짝이 0이면 없음, 0보다 크면 모름(최빈값으로 채운다).
def fill_missing(train, test):
    pool = pd.concat([train, test], ignore_index=True)
    nb_lotfrontage = pool.groupby('Neighborhood')['LotFrontage'].median()
    fills, out = [], []

    for name, df in (('train', train), ('test', test)):
        df = df.copy()

        def fill(mask, col, value, rule):
            for idx in df.index[mask]:
                v = value[idx] if isinstance(value, pd.Series) else value
                fills.append(dict(dataset=name, Id=int(df.at[idx, 'Id']), column=col, value=v, rule=rule))
                df.at[idx, col] = v

        no_bsmt = df[BSMT_CATS].isna().all(axis=1) & (df['TotalBsmtSF'].isna() | (df['TotalBsmtSF'] == 0))
        for col in BSMT_NUMS:
            fill(no_bsmt & df[col].isna(), col, 0.0, '시설 없음 → 0')
        fill(df['GarageType'].isna() & df['GarageYrBlt'].isna(), 'GarageYrBlt', df['YearBuilt'], '차고 없음 → YearBuilt')
        for col in GARAGE_NUMS:
            fill(df['GarageType'].notna() & df[col].isna(), col, pool[col].median(), '차고 있음 → 중앙값')
        fill(df['LotFrontage'].isna(), 'LotFrontage', df['Neighborhood'].map(nb_lotfrontage), '지역별 중앙값')
        for col in numeric_columns(df):
            fill(df[col].isna(), col, pool[col].median(), '중앙값')

        for col, value in FIXED_FILL.items():
            fill(df[col].isna(), col, value, '비고의 값')
        for col in NA_IS_NONE:
            fill(df[col].isna(), col, 'NA', '없음')
        for col, (pair, none_value) in PAIR.items():
            fill(df[col].isna() & (df[pair] == 0), col, none_value, f'{pair} = 0 → 없음')
            missing = df[col].isna() & (df[pair] > 0)
            if not missing.any():
                continue
            if col == 'MasVnrType':
                fill(missing, col, MASVNR_TYPE_WITH_AREA, f'{pair} > 0 → BrkFace (사용자 결정, 3절 7번과 같게)')
                continue
            mode, modes = _mode(pool[col])
            tie = f' (동률 {"·".join(modes)} → {mode})' if len(modes) > 1 else ''
            fill(missing, col, mode, f'{pair} > 0 → 최빈값{tie}')
        out.append(df)

    fills = pd.DataFrame(fills, columns=['dataset', 'Id', 'column', 'value', 'rule'])
    return out[0], out[1], fills

## Columns — categorical encoding / 열 — 범주형 인코딩

### Three routes for 43 string columns / 글자 열 43개의 세 갈래

범주형 피쳐에 명확한 순서가 있으면 그것을 사용하고, 순서가 없는 경우 OOF 타깃 인코딩을 사용한다.<br>
<br>
<div style="text-align:left">
<table style="display:inline-table; margin:0 !important">
<thead><tr><th style="text-align:left">길</th><th style="text-align:center">열 수</th><th style="text-align:left">이유</th></tr></thead>
<tbody>
<tr><td style="text-align:left"><b>순서형 정수</b></td><td style="text-align:center">23</td><td style="text-align:left">방어 가능한 순서가 있다(<code>Ex &gt; Gd &gt; TA &gt; Fa &gt; Po</code>). 1열이고 추정 잡음도 누출도 없다</td></tr>
<tr><td style="text-align:left"><b>파생 그룹</b></td><td style="text-align:center">9 → 21</td><td style="text-align:left">수준이 너무 많고, 도메인 묶음이 그보다 낫다(외장재 계열, 동네 평당가)</td></tr>
<tr><td style="text-align:left"><b>타깃 인코딩</b></td><td style="text-align:center">11</td><td style="text-align:left">방어 가능한 순서가 없으니 정답이 순서를 공급한다</td></tr>
</tbody></table>
</div><br>
순서형 23개를 타깃 인코딩으로 바꾸면 알려진 구조를 추정 잡음으로 교체하는 셈이 된다.

In [10]:
# Stage 3 — one route per column, chosen by what the column's own structure allows.
# 3단계 — 열마다 그 열의 구조가 허락하는 길로 보낸다.
#   ORDINAL   : 품질 척도를 정수 순서로 (Ex=5 ... Po=1)
#   DECOMPOSE : 한 열을 뜻이 다른 여러 0/1 열로 분해 (HouseStyle → 층수·반층·상층마감)
#   EXTERIOR  : 외장재 15종 → 재료 계열 5개 + '혼합' 표시
#   ONEHOT    : 순서가 없는 9열. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(GarageType·MasVnrType까지 11열)
#   UNIT_PRICE: 동네별 평당가. 유일하게 가격을 쓰는 열이고, 학습 행에서는 자기 행을 빼고 계산한다

QUALITY = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}


# 23 columns have a defensible order, so they become integers. One column, no estimation noise, no leakage.
# 23개 열은 방어 가능한 순서가 있어 정수로 둔다. 1열이고 추정 잡음도 누출도 없다.
ORDINAL = {
    'ExterQual': dict(QUALITY),
    'ExterCond': dict(QUALITY),
    'BsmtQual': {'NA': 0, **QUALITY},
    'BsmtCond': {'NA': 0, **QUALITY},
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': dict(QUALITY),
    'KitchenQual': dict(QUALITY),
    'FireplaceQu': {'NA': 0, **QUALITY},
    'GarageType': {'NA': 0, 'CarPort': 1, 'Detchd': 2, 'Attchd': 3, 'Basment': 3, '2Types': 3, 'BuiltIn': 4},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'NA': 0, **QUALITY},
    'GarageCond': {'NA': 0, **QUALITY},
    'PoolQC': {'NA': 0, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'MasVnrType': {'None': 0, 'BrkCmn': 1, 'BrkFace': 2, 'Stone': 3},
    'Alley': {'NA': 0, 'Grvl': 1, 'Pave': 2},
    'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
    'LotShape': {'Reg': 1, 'IR1': 2, 'IR2': 3, 'IR3': 4},
    'LandSlope': {'Gtl': 1, 'Mod': 2, 'Sev': 3},
    'PavedDrive': {'N': 1, 'P': 2, 'Y': 3},
    'Electrical': {'FuseP': 1, 'FuseF': 2, 'FuseA': 3, 'SBrkr': 4, 'Mix': 2},
    'Utilities': {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4},
    'OverallQual': {i: i for i in range(1, 11)},
    'OverallCond': {i: i for i in range(1, 11)},
}

NONE_CODES = ('NA', 'None')

BINARY = {
    'CentralAir': {'N': 0, 'Y': 1},
    'Street': {'Grvl': 0, 'Pave': 1},
}

_HOUSE_STYLES = ['1Story', '1.5Fin', '1.5Unf', '2Story', '2.5Fin', '2.5Unf', 'SFoyer', 'SLvl']

_BLDG_TYPES = ['1Fam', '2fmCon', 'Duplex', 'TwnhsE', 'Twnhs']

_LOT_CONFIGS = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']

_CONDITIONS = ['Norm', 'Feedr', 'Artery', 'RRNn', 'RRNe', 'RRAn', 'RRAe', 'PosN', 'PosA']

_FENCES = ['NA', 'GdPrv', 'MnPrv', 'GdWo', 'MnWw']

def _rest(codes, value, **explicit):
    return {c: explicit.get(c, value) for c in codes}


# 뜻이 섞인 열을 속성별 0/1 열로 분해한다. HouseStyle → 층수·반층·상층마감.
DECOMPOSE = {
    'HouseStyle': {
        'HouseStyle_Floors': ('층수', {'1Story': 1, '1.5Fin': 1.5, '1.5Unf': 1.5, '2Story': 2, '2.5Fin': 2.5,
                                      '2.5Unf': 2.5, 'SFoyer': 1.5, 'SLvl': 1.5}),
        'HouseStyle_UpperFin': ('위층 마감', _rest(_HOUSE_STYLES, 1, **{'1.5Unf': 0, '2.5Unf': 0})),
        'HouseStyle_Split': ('스플릿', _rest(_HOUSE_STYLES, 0, SFoyer=1, SLvl=1)),
    },
    'BldgType': {
        'BldgType_Units': ('가구 수', {'1Fam': 1, 'TwnhsE': 1, 'Twnhs': 1, 'Duplex': 2, '2fmCon': 2}),
        'BldgType_SharedWalls': ('타운하우스 공유 벽 수', {'1Fam': 0, '2fmCon': 0, 'Duplex': 0, 'TwnhsE': 1, 'Twnhs': 2}),
        'BldgType_Conversion': ('개조', _rest(_BLDG_TYPES, 0, **{'2fmCon': 1})),
    },
    'LotConfig': {
        'LotConfig_Frontages': ('도로 접면 수', {'Inside': 1, 'CulDSac': 1, 'Corner': 2, 'FR2': 2, 'FR3': 3}),
        'LotConfig_Corner': ('코너', _rest(_LOT_CONFIGS, 0, Corner=1)),
        'LotConfig_CulDSac': ('막다른 길', _rest(_LOT_CONFIGS, 0, CulDSac=1)),
    },
    'Condition': {
        'Condition_Road': ('큰 도로', _rest(_CONDITIONS, 0, Feedr=1, Artery=2)),
        'Condition_Rail': ('철도', _rest(_CONDITIONS, 0, RRNn=1, RRNe=1, RRAn=2, RRAe=2)),
        'Condition_Pos': ('긍정 요소', _rest(_CONDITIONS, 0, PosN=1, PosA=2)),
    },
    'Fence': {
        'Fence_Privacy': ('프라이버시 울타리', _rest(_FENCES, 0, MnPrv=1, GdPrv=2)),
        'Fence_Wood': ('목재 울타리', _rest(_FENCES, 0, MnWw=1, GdWo=2)),
    },
}

DECOMPOSE_SOURCES = {'HouseStyle': ['HouseStyle'], 'BldgType': ['BldgType'], 'LotConfig': ['LotConfig'],
                     'Condition': ['Condition1', 'Condition2'], 'Fence': ['Fence']}


# 외장재 15종을 재료 계열 5개로 묶는다. 비닐이 기준이고 두 면이 다르면 혼합 표시를 켠다.
EXTERIOR_FAMILY = {
    'VinylSd': 'Vinyl',
    'MetalSd': 'Metal',
    'Wd Sdng': 'Wood', 'WdShing': 'Wood', 'Plywood': 'Wood', 'HdBoard': 'Wood',
    'CemntBd': 'CementStucco', 'Stucco': 'CementStucco', 'ImStucc': 'CementStucco',
    'BrkFace': 'BrickStone', 'BrkComm': 'BrickStone', 'Stone': 'BrickStone', 'CBlock': 'BrickStone',
    'AsbShng': 'ShingleOther', 'AsphShn': 'ShingleOther', 'Other': 'ShingleOther', 'PreCast': 'ShingleOther',
}

EXTERIOR_BASE = 'Vinyl'

EXTERIOR_COLUMNS = {
    'Exterior_Metal': ('금속', 'Metal'),
    'Exterior_Wood': ('목재', 'Wood'),
    'Exterior_CementStucco': ('시멘트·스터코', 'CementStucco'),
    'Exterior_BrickStone': ('벽돌·석재', 'BrickStone'),
    'Exterior_ShingleOther': ('슁글·기타', 'ShingleOther'),
}

EXTERIOR_MIXED = ('Exterior_Mixed', '외장재 혼합')

EXTERIOR2_SPELLING = {'CmentBd': 'CemntBd', 'Wd Shng': 'WdShing', 'Brk Cmn': 'BrkComm'}


# These 9 columns have no defensible order. The 0/1 columns made here are dropped and target-encoded instead.
# 이 9개 열은 순서가 없다. 여기서 만든 0/1 열은 버리고 타깃 인코딩으로 대체한다(아래 FACTOR_COLUMNS).
ONEHOT = {
    'MSZoning': [('MSZoning_RM', 'RM', ['RM']), ('MSZoning_RH', 'RH', ['RH']), ('MSZoning_FV', 'FV', ['FV']),
                 ('MSZoning_C', '상업', ['C (all)'])],
    'LandContour': [('LandContour_Bnk', '둔덕', ['Bnk']), ('LandContour_HLS', '언덕 사면', ['HLS']),
                    ('LandContour_Low', '저지대', ['Low'])],
    'RoofStyle': [('RoofStyle_Hip', '모임지붕', ['Hip']),
                  ('RoofStyle_Other', '기타 지붕 형태', ['Flat', 'Gambrel', 'Mansard', 'Shed'])],
    'RoofMatl': [('RoofMatl_Wood', '목재 지붕', ['WdShngl', 'WdShake']),
                 ('RoofMatl_Other', '기타 지붕 재료', ['Tar&Grv', 'Metal', 'Membran', 'Roll', 'ClyTile'])],
    'Foundation': [('Foundation_PConc', '타설 콘크리트', ['PConc']), ('Foundation_CBlock', '콘크리트 블록', ['CBlock']),
                   ('Foundation_BrkTil', '벽돌·타일', ['BrkTil'])],
    'Heating': [('Heating_HotWater', '온수·증기', ['GasW', 'OthW']),
                ('Heating_OldFurnace', '구식 난로', ['Grav', 'Wall', 'Floor'])],
    'MiscFeature': [('MiscFeature_Shed', '창고', ['Shed']),
                    ('MiscFeature_Other', '기타 시설', ['Gar2', 'Othr', 'TenC', 'Elev'])],
    'SaleType': [('SaleType_New', '신축 판매', ['New']), ('SaleType_Contract', '계약 판매', ['ConLD', 'ConLI', 'ConLw', 'Con']),
                 ('SaleType_COD', '법원·유산', ['COD'])],
    'SaleCondition': [('SaleCondition_Partial', '미완공 신축', ['Partial']),
                      ('SaleCondition_Abnorml', '비정상 거래', ['Abnorml']),
                      ('SaleCondition_Special', '특수 거래', ['Family', 'Alloca', 'AdjLand'])],
}

ONEHOT_BASE = {'MSZoning': ['RL'], 'LandContour': ['Lvl'], 'RoofStyle': ['Gable'], 'RoofMatl': ['CompShg'],
               'Foundation': ['Slab', 'Stone', 'Wood'], 'Heating': ['GasA'], 'MiscFeature': ['NA'],
               'SaleType': ['WD', 'CWD', 'VWD', 'Oth'], 'SaleCondition': ['Normal']}


# The only column that uses the price. Per-neighborhood unit price, and the training row's own value is left out.
# 가격을 쓰는 유일한 열. 동네별 평당가이고, 학습 행에서는 자기 행을 뺀 값을 쓴다.
UNIT_PRICE = 'Neighborhood_UnitPrice'


# MSSubClass는 HouseStyle + BldgType + 건축연도로 98.4% 복원되어 지운다.
DROP = ['MSSubClass']

NUMERIC = ['LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
           'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
           'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
           'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
           'MiscVal', 'MoSold', 'YrSold']

CATEGORICAL_FEATURES = (list(ORDINAL) + list(BINARY)
                        + [col for group in DECOMPOSE.values() for col in group]
                        + list(EXTERIOR_COLUMNS) + [EXTERIOR_MIXED[0]]
                        + [col for cols in ONEHOT.values() for col, _, _ in cols]
                        + [UNIT_PRICE])

FEATURES = CATEGORICAL_FEATURES + NUMERIC

FIXED_FEATURES = [f for f in FEATURES if f != UNIT_PRICE]

SOURCE_COLUMNS = (list(ORDINAL) + list(BINARY) + [c for cols in DECOMPOSE_SOURCES.values() for c in cols]
                  + ['Exterior1st', 'Exterior2nd'] + list(ONEHOT) + ['Neighborhood'] + DROP)

def _map(series, mapping, name, numeric=True):
    unknown = set(series.dropna().unique()) - set(mapping)
    if unknown:
        raise ValueError(f'{name}: 매핑에 없는 값 {sorted(map(str, unknown))}')
    mapped = series.map(mapping)
    return mapped.astype(float) if numeric else mapped


# 가격을 쓰지 않는 인코딩 열과 수치형 열을 만든다. Id·Neighborhood는 함께 넘긴다.
def encode_fixed(df):
    cols = {}
    for col, mapping in {**ORDINAL, **BINARY}.items():
        cols[col] = _map(df[col], mapping, col)
    for group, attrs in DECOMPOSE.items():
        sources = DECOMPOSE_SOURCES[group]
        for new, (_, mapping) in attrs.items():
            values = [_map(df[s], mapping, f'{s}→{new}') for s in sources]
            cols[new] = values[0] if len(values) == 1 else np.maximum(*values)
    family1 = _map(df['Exterior1st'], EXTERIOR_FAMILY, 'Exterior1st', numeric=False)
    family2 = _map(df['Exterior2nd'].replace(EXTERIOR2_SPELLING), EXTERIOR_FAMILY, 'Exterior2nd', numeric=False)
    for new, (_, family) in EXTERIOR_COLUMNS.items():
        cols[new] = (family1 == family).astype(float)
    cols[EXTERIOR_MIXED[0]] = (family1 != family2).astype(float)
    for col, groups in ONEHOT.items():
        known = set(ONEHOT_BASE[col]) | {v for _, _, members in groups for v in members}
        unknown = set(df[col].dropna().unique()) - known
        if unknown:
            raise ValueError(f'{col}: 매핑에 없는 값 {sorted(unknown)}')
        for new, _, members in groups:
            cols[new] = df[col].isin(members).astype(float).where(df[col].notna())
    for col in NUMERIC:
        cols[col] = df[col].astype(float)
    keep = [c for c in ('Id', 'Neighborhood', 'SalePrice') if c in df.columns]
    out = pd.concat([df[keep], pd.DataFrame(cols, index=df.index)], axis=1)
    return out[keep + FIXED_FEATURES]

## Folds — leakage control and the design matrix / fold — 누출 통제와 설계행렬

### Leakage control / 누출 통제

In [11]:
# Stage 4 — leakage control. Every answer-touching statistic is fitted on the fold's train rows only.
# 4단계 — 누출 통제. 정답을 건드리는 통계는 그 fold의 학습 행으로만 적합한다.
#   FoldPreprocessor.fit(train)      : 평당가·스케일·타깃 통계를 학습 행에서 구한다
#   FoldPreprocessor.transform(other): valid·test 행은 가격을 **읽지 않는다**
#   make_folds                       : fold 배정. 이 노트북은 공유된 fold 표를 쓰고, 이 함수와 일치하는지 검증한다

SEED = 42

N_FOLDS = 5


# 타깃 인코딩의 평활값. 희귀 수준이 전체 평균 쪽으로 끌려간다.
SMOOTHING_M = 10


# 0에 뜻이 없고 값이 0에서 멀리 떨어진 열만 z-score한다. 나머지는 표준편차로만 나눈다.
ZSCORE = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold', 'MoSold', 'LotFrontage']

def scale_method(feature):
    return 'zscore' if feature in ZSCORE else 'divide'


# fit() sees the fold's training rows only; transform() never reads a price. This is the leakage boundary.
# fit()은 그 fold의 학습 행만 본다. transform()은 가격을 읽지 않는다. 누출의 경계가 여기다.
class FoldPreprocessor:

    def fit(self, enc):
        price = enc['SalePrice'].to_numpy(float)
        unit = price / enc['GrLivArea'].to_numpy(float)
        groups = pd.Series(unit).groupby(enc['Neighborhood'].to_numpy())
        self.unit_mean_ = float(unit.mean())
        self.unit_sum_ = groups.sum().to_dict()
        self.unit_count_ = groups.count().to_dict()
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        loo = (S - unit + m * mu) / (n - 1 + m)
        raw = self._assemble(enc, loo)
        self.mean_ = raw.mean(axis=0)
        std = raw.std(axis=0)
        self.std_ = np.where(std > 0, std, 1.0)
        self.method_ = np.array([scale_method(f) for f in FEATURES])
        self.center_ = np.where(self.method_ == 'zscore', self.mean_, 0.0)
        log_price = np.log(price)
        self.target_mean_ = float(log_price.mean())
        self.target_std_ = float(log_price.std())
        self.n_train_ = len(enc)
        self.X_train_ = (raw - self.center_) / self.std_
        return self

    def transform(self, enc):
        S, n = self._neighborhood_stats(enc)
        m, mu = SMOOTHING_M, self.unit_mean_
        return (self._assemble(enc, (S + m * mu) / (n + m)) - self.center_) / self.std_

    def target(self, sale_price):
        return (np.log(np.asarray(sale_price, float)) - self.target_mean_) / self.target_std_

    def to_log_price(self, z):
        return np.asarray(z) * self.target_std_ + self.target_mean_

    def scale_table(self):
        return pd.DataFrame({'feature': FEATURES, 'method': self.method_, 'mean': self.mean_, 'std': self.std_})

    def _neighborhood_stats(self, enc):
        nb = enc['Neighborhood']
        S = nb.map(self.unit_sum_).fillna(0.0).to_numpy(float)
        n = nb.map(self.unit_count_).fillna(0).to_numpy(float)
        return S, n

    @staticmethod
    def _assemble(enc, unit_price):
        raw = pd.DataFrame(enc[FIXED_FEATURES].to_numpy(float), columns=FIXED_FEATURES)
        raw[UNIT_PRICE] = unit_price
        raw = raw[FEATURES].to_numpy(float)
        if np.isnan(raw).any():
            bad = sorted({FEATURES[j] for j in np.where(np.isnan(raw))[1]})
            raise ValueError(f'재료에 빈 칸이 있다(사용자 결정 대기일 수 있다): {bad}')
        return raw


# fold 배정. 이 노트북은 공유된 fold 표를 쓰고 이 함수와 일치하는지 검증한다.
def make_folds(n_rows, n_folds=N_FOLDS, seed=SEED):
    fold = np.empty(n_rows, dtype=int)
    for k, (_, valid_idx) in enumerate(KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(np.arange(n_rows))):
        fold[valid_idx] = k
    return fold

@dataclass
class FoldData:
    fold: int
    X_train: np.ndarray
    y_train: np.ndarray
    X_valid: np.ndarray
    y_valid: np.ndarray
    target_mean: float
    target_std: float
    ids_train: np.ndarray
    ids_valid: np.ndarray


# fold k를 valid로 두고 나머지로 전처리를 적합한다.
def fold_data(enc, folds, k):
    train, valid = enc[folds != k].reset_index(drop=True), enc[folds == k].reset_index(drop=True)
    pre = FoldPreprocessor().fit(train)
    return FoldData(k, pre.X_train_, np.log(train['SalePrice'].to_numpy(float)), pre.transform(valid),
                    np.log(valid['SalePrice'].to_numpy(float)), pre.target_mean_, pre.target_std_,
                    train['Id'].to_numpy(), valid['Id'].to_numpy()), pre


# 학습 행 전체로 적합한다(fold를 나누지 않는 최종 실행용).
def full_data(enc):
    pre = FoldPreprocessor().fit(enc)
    return FoldData(-1, pre.X_train_, np.log(enc['SalePrice'].to_numpy(float)), None, None,
                    pre.target_mean_, pre.target_std_, enc['Id'].to_numpy(), None), pre

### Out-of-fold target encoding / 아웃오브폴드 타깃 인코딩

타깃 인코딩은 수준을 그 수준의 평균 정답으로 바꾼다. 그 행의 정답이 그 행의 피쳐 안에 들어간다.<br>
<br>
LOO로 고치려 들면 더 나빠진다. 수준 합을 $S_l$, 수준 행수를 $n_l$이라 하면<br>
<br>
$$\text{LOO}_i = \frac{S_l - y_i + m\mu}{n_l - 1 + m} = -\frac{y_i}{n_l - 1 + m} + \text{const}_l$$
<br>
그 행 정답의 완전한 아핀 함수다. 수준을 식별하고 상수항을 걷어내면 $y_i$가 그대로 나온다. 실제로 재 보면
상관이 −1.0이다. 이 열을 GP에 주면 train은 0에 가깝게 내려가는데 valid는 0.4를 넘고, GCV는 끝까지 그것을
못 본다. 누출이 적합 단계가 아니라 그 위의 열에 있기 때문이다.<br>
<br>
아웃오브폴드는 학습 행을 내부 5 fold으로 나누고 각 행에 나머지 네 fold의 통계를 준다. 자기 정답이 자기
피쳐에 들어가지 않으니 역산할 대상이 없다.<br>
<br>
수준 상수 방식과도 다르다. 그쪽은 자기 행이 자기 수준 평균에 $1/(n_l+m)$만큼 섞여서 학습 쪽이 낙관적이다.
아웃오브폴드는 학습 행도 '자기가 안 들어간 평균'을 보므로 valid 행과 같은 처지가 된다.

In [12]:
# Stage 5 — terminals and the design matrix. 79 engineered + 11 out-of-fold target encodings = 90.
# 5단계 — 터미널과 설계행렬. 가공 79열 + 아웃오브폴드 타깃 인코딩 11열 = 90개.
#   FACTOR_COLUMNS   : 순서가 없어 타깃 인코딩으로 보내는 11열
#   KEPT             : 그 11열의 인코딩을 뺀 나머지 79열
#   target_encode_oof: 아웃오브폴드 타깃 인코딩. 위 설명의 그 함수다
#   build            : fold 하나의 Design(학습·검증·test 행렬과 Id)을 만든다

OOF_FOLDS = 5

TE_ENCODINGS = ('target', 'loo', 'oof', 'mixed')

FACTOR_ONEHOT = list(ONEHOT)

FACTOR_ORDINAL = ['GarageType', 'MasVnrType']


# The 11 nominal columns with no defensible order. These get the out-of-fold target encoding.
# 순서가 없는 명목형 11열. 이 열들이 아웃오브폴드 타깃 인코딩을 받는다.
FACTOR_COLUMNS = FACTOR_ONEHOT + FACTOR_ORDINAL

DROPPED = [new for col in FACTOR_ONEHOT for new, _, _ in ONEHOT[col]] + FACTOR_ORDINAL


# 터미널이 되는 79열. 위 11열의 인코딩을 뺀 나머지다.
KEPT = [f for f in FEATURES if f not in set(DROPPED)]


# 컬럼의 수준 목록. 드문 값 병합은 원래 인코딩 규칙을 그대로 따른다.
def factor_levels(col):
    if col in ONEHOT:
        groups = [list(ONEHOT_BASE[col])] + [list(vals) for _, _, vals in ONEHOT[col]]
        return groups
    mapping = ORDINAL[col]
    codes = sorted(set(mapping.values()))
    return [[k for k, v in mapping.items() if v == code] for code in codes]

def level_index(col):
    return {v: i for i, group in enumerate(factor_levels(col)) for v in group}


# theta 초기값. 이 노트북에서는 쓰지 않는다.
def theta_base(col, std):
    if col in ONEHOT:
        n = len(factor_levels(col))
        return (np.arange(n, dtype=float) - (n - 1) / 2) / max(n - 1, 1) * 2
    codes = np.array(sorted(set(ORDINAL[col].values())), dtype=float)
    return codes / (std if std > 0 else 1.0)


# The LOO variant is kept only to show why it is unusable: it is an exact affine function of the row's own answer.
# LOO 변형은 '왜 쓸 수 없는지'를 보이기 위해서만 남겼다. 그 행 정답의 완전한 아핀 함수다(상관 -1.0).
# 수준 상수 방식은 자기 행이 자기 수준 평균에 1/(n+m)만큼 섞여 학습 쪽이 낙관적이다.
def target_encode(codes_tr, y_tr, n_lv, codes_other=None, m=SMOOTHING_M, loo=False):
    mu = float(y_tr.mean())
    S = np.bincount(codes_tr, weights=y_tr, minlength=n_lv)
    n = np.bincount(codes_tr, minlength=n_lv).astype(float)
    lvl = (S + m * mu) / (n + m)
    tr = ((S[codes_tr] - y_tr + m * mu) / (n[codes_tr] - 1 + m)) if loo else lvl[codes_tr]
    return tr, (lvl[codes_other] if codes_other is not None else None), lvl


# Out-of-fold: a training row sees the statistic of the four inner folds it does not belong to.
# 아웃오브폴드. 학습 행은 자기가 속하지 않은 내부 네 fold의 통계를 본다.
# 자기 정답이 자기 피쳐에 들어가지 않으므로 역산할 대상이 없다. valid·test 행은 학습 전체 통계를 쓴다.
def target_encode_oof(codes_tr, y_tr, n_lv, inner, codes_other=None, m=SMOOTHING_M):
    tr = np.empty(len(codes_tr), float)
    for j in np.unique(inner):
        ref = inner != j
        y_ref, c_ref = y_tr[ref], codes_tr[ref]
        mu_j = float(y_ref.mean())
        S_j = np.bincount(c_ref, weights=y_ref, minlength=n_lv)
        n_j = np.bincount(c_ref, minlength=n_lv).astype(float)
        lvl_j = (S_j + m * mu_j) / (n_j + m)
        tr[inner == j] = lvl_j[codes_tr[inner == j]]
    mu = float(y_tr.mean())
    S = np.bincount(codes_tr, weights=y_tr, minlength=n_lv)
    n = np.bincount(codes_tr, minlength=n_lv).astype(float)
    lvl = (S + m * mu) / (n + m)
    return tr, (lvl[codes_other] if codes_other is not None else None), lvl


# fold 하나의 재료. 학습·검증·test 행렬과 Id, 그리고 터미널 이름이 들어 있다.
@dataclass
class Design:
    fold: int
    X_train: np.ndarray
    y_train: np.ndarray
    codes_train: dict
    X_valid: np.ndarray
    y_valid: np.ndarray
    codes_valid: dict
    theta_init: dict
    n_levels: dict
    features: list
    ids_train: np.ndarray
    ids_valid: np.ndarray
    encoding: str = 'theta'
    arg_names: list = None
    n_cols: int = None
    loo_levels: dict = None
    X_holdout: np.ndarray = None
    y_holdout: np.ndarray = None
    codes_holdout: dict = None
    ids_holdout: np.ndarray = None
    X_test: np.ndarray = None
    codes_test: dict = None
    ids_test: np.ndarray = None

def _codes(raw, cols):
    out = {}
    for col in cols:
        idx = level_index(col)
        src = raw['Condition1'] if col == 'Condition' else raw[col]
        out[col] = src.map(idx).to_numpy(np.int64)
        assert not np.any(out[col] < 0), col
    return out


# One fold's design matrix: 79 engineered columns + 11 out-of-fold target encodings = 90 terminals.
# fold 하나의 설계행렬. 가공 79열 + 아웃오브폴드 타깃 인코딩 11열 = 터미널 90개.
def build(fold=None, folds=None, prepared=None, encoding='theta', holdout=False, test=False,
          inner_seed=None, drop_ids=()):
    p = prepared or prepare()
    ts = p.train_splitted
    enc_all = encode_fixed(ts)
    keep = [FEATURES.index(f) for f in KEPT]
    if drop_ids:
        m = ~ts['Id'].isin(list(drop_ids)).to_numpy()
        if folds is None and fold is not None:
            folds = make_folds(len(enc_all))
        if folds is not None:
            folds = np.asarray(folds)[m]
        ts, enc_all = ts[m].reset_index(drop=True), enc_all[m].reset_index(drop=True)
    if fold is None and inner_seed is None:
        fd, pre = full_data(enc_all)
        tr_raw, va_raw = ts, None
    else:
        if fold is None:
            f, k = make_folds(len(enc_all), seed=inner_seed), 0
        else:
            f = folds if folds is not None else make_folds(len(enc_all))
            k = fold
        fd, pre = fold_data(enc_all, f, k)
        tr_raw = ts[f != k].reset_index(drop=True)
        va_raw = ts[f == k].reset_index(drop=True)
    std = {c: float(pre.std_[FEATURES.index(c)]) for c in FACTOR_ORDINAL}
    Xtr = np.ascontiguousarray(fd.X_train[:, keep])
    Xva = None if fd.X_valid is None else np.ascontiguousarray(fd.X_valid[:, keep])
    codes_tr = _codes(tr_raw, FACTOR_COLUMNS)
    codes_va = None if va_raw is None else _codes(va_raw, FACTOR_COLUMNS)
    n_lv = {c: len(factor_levels(c)) for c in FACTOR_COLUMNS}
    names = list(KEPT)
    loo_lvl = None

    if encoding in TE_ENCODINGS:
        inner = (make_folds(len(fd.y_train), n_folds=OOF_FOLDS) if encoding == 'oof' else None)
        extra_tr, extra_va, loo_lvl = [], [], {}
        for c in FACTOR_COLUMNS:
            other = None if codes_va is None else codes_va[c]
            if encoding == 'oof':
                t, v, lvl = target_encode_oof(codes_tr[c], fd.y_train, n_lv[c], inner, other)
            else:
                t, v, lvl = target_encode(codes_tr[c], fd.y_train, n_lv[c], other,
                                          loo=(encoding == 'loo'))
            mu, sd = float(t.mean()), float(t.std())
            sd = sd if sd > 1e-12 else 1.0
            extra_tr.append((t - mu) / sd)
            if v is not None:
                extra_va.append((v - mu) / sd)
            loo_lvl[c] = dict(levels=(lvl - mu) / sd, mean=mu, std=sd, raw_levels=lvl)
            names.append(f'{c}_TE')
        Xtr = np.ascontiguousarray(np.column_stack([Xtr] + extra_tr))
        if Xva is not None:
            Xva = np.ascontiguousarray(np.column_stack([Xva] + extra_va))
        if encoding == 'mixed':
            arg_names = list(names) + [f'CAT_{c}' for c in FACTOR_COLUMNS]
            n_cols = len(names)
        else:
            codes_tr, codes_va, n_lv = {}, ({} if codes_va is not None else None), {}
            arg_names, n_cols = list(names), len(names)
    elif encoding == 'numeric':
        codes_tr, codes_va, n_lv = {}, ({} if codes_va is not None else None), {}
        arg_names, n_cols = list(KEPT), len(KEPT)
    else:
        arg_names = list(KEPT) + [f'FAC_{c}' for c in FACTOR_COLUMNS]
        n_cols = len(KEPT)

    def _block(frame):
        src = frame if 'SalePrice' in frame.columns else frame.assign(SalePrice=1.0)
        X = np.ascontiguousarray(pre.transform(encode_fixed(src))[:, keep])
        cd = _codes(frame, FACTOR_COLUMNS)
        if encoding in TE_ENCODINGS:
            X = np.ascontiguousarray(np.column_stack(
                [X] + [loo_lvl[c]['levels'][cd[c]] for c in FACTOR_COLUMNS]))
            if encoding != 'mixed':
                cd = {}
        return X, cd

    Xho = yho = codes_ho = ids_ho = None
    if holdout:
        Xho, codes_ho = _block(p.holdout)
        yho = np.log(p.holdout['SalePrice'].to_numpy(float))
        ids_ho = p.holdout['Id'].to_numpy()
    Xte = codes_te = ids_te = None
    if test:
        Xte, codes_te = _block(p.test)
        ids_te = p.test['Id'].to_numpy()

    return Design(
        fold=-1 if fold is None else fold,
        X_train=Xtr, y_train=fd.y_train, codes_train=codes_tr,
        X_valid=Xva, y_valid=fd.y_valid, codes_valid=codes_va,
        theta_init=({} if encoding in TE_ENCODINGS or encoding == 'numeric' else
                    {c: theta_base(c, std.get(c, 1.0)) for c in FACTOR_COLUMNS}),
        n_levels=n_lv, features=names, ids_train=fd.ids_train, ids_valid=fd.ids_valid,
        encoding=encoding, arg_names=arg_names, n_cols=n_cols, loo_levels=loo_lvl,
        X_holdout=Xho, y_holdout=yho, codes_holdout=codes_ho, ids_holdout=ids_ho,
        X_test=Xte, codes_test=codes_te, ids_test=ids_te)

### Build the design matrices / 설계행렬 생성

In [13]:
# Every usable training row goes into the five folds. Nothing is held out.
# 쓸 수 있는 학습 행 전부를 fold 다섯 개에 넣는다. 따로 떼어 두는 행은 없다.
from types import SimpleNamespace

train_fixed, _changes = fix_contradictions(train_csv, test_csv)[:2], None
train_fixed, test_fixed, CHANGES = fix_contradictions(train_csv, test_csv)
train_fixed, test_fixed, FILLS = fill_missing(train_fixed, test_fixed)

PREP = SimpleNamespace(train=train_fixed, train_splitted=train_fixed,
                       holdout=train_fixed.iloc[:0], test=test_fixed)
FOLDS = train_fixed.Id.map(FOLD_OF).to_numpy()
assert not pd.isna(FOLDS).any(), 'a training row has no fold number!!'
FOLDS = FOLDS.astype(int)

DES = {f: build(fold=f, folds=FOLDS, prepared=PREP, encoding='oof', test=True) for f in range(5)}
print(f'rows fixed {len(train_fixed):,} · value changes {len(CHANGES)} · fills {len(FILLS)}')
for f in range(5):
    d = DES[f]
    print(f'  fold {f}: train {len(d.y_train):,} · valid {len(d.y_valid):,} '
          f'· terminals {d.X_train.shape[1]} · test {len(d.ids_test):,}')
print(f'max RSS {RSS_MB():.0f} MB')

rows fixed 1,453 · value changes 6 · fills 13933
  fold 0: train 1,162 · valid 291 · terminals 90 · test 1,459
  fold 1: train 1,162 · valid 291 · terminals 90 · test 1,459
  fold 2: train 1,162 · valid 291 · terminals 90 · test 1,459
  fold 3: train 1,163 · valid 290 · terminals 90 · test 1,459
  fold 4: train 1,163 · valid 290 · terminals 90 · test 1,459
max RSS 265 MB


### Preprocessing sanity check / 전처리 정합성 검사

In [14]:
# Check 1 of 4 — the data. Every assert here is against the data itself, not a number I measured earlier.
# 검증 1/4 — **데이터**. 여기의 모든 assert는 외부 기록이 아니라 데이터 자신을 근거로 한다.
_dropped = sorted(int(i) for i in set(train_csv.Id) - set(train_fixed.Id))
assert len(_dropped) == 7, f'{len(_dropped)} rows were dropped, expected 7: {_dropped}!!'
assert set(train_fixed.Id) == set(fold_tbl.Id), 'the fixed training rows and the fold table differ!!'

# The shared fold table has to be what make_folds produces here — this is where a sklearn version
# difference would show up, and it would change every fold silently.
# 공유된 fold 표는 이 환경의 make_folds 결과와 같아야 한다. sklearn 판 차이가 드러나는 자리고,
# 어긋나면 fold 전체가 조용히 달라진다.
assert np.array_equal(make_folds(len(train_fixed)), FOLDS), \
    'make_folds disagrees with the shared fold table (sklearn version drift?)!!'

print(f'contradiction rows removed {_dropped}')
print(f'fold sizes {np.bincount(FOLDS).tolist()} · make_folds agrees with the shared table')

print(f'\n{"fold":>5}{"train":>7}{"valid":>7}{"test":>7}{"terminals":>11}{"y sum":>13}')
for f in range(5):
    d = DES[f]
    assert d.X_train.shape[1] == 90, f'fold {f} has {d.X_train.shape[1]} terminals, not 90!!'
    assert len(d.y_train) + len(d.y_valid) == len(train_fixed), f'fold {f} rows do not add up!!'
    assert np.isfinite(d.X_train).all() and np.isfinite(d.X_valid).all(), f'fold {f} non-finite!!'
    assert np.isfinite(d.X_test).all(), f'fold {f} test matrix has non-finite values!!'
    assert not (set(d.ids_train) & set(d.ids_valid)), f'fold {f} train and valid share a row!!'
    print(f'{f:>5}{len(d.y_train):>7,}{len(d.y_valid):>7,}{len(d.ids_test):>7,}'
          f'{d.X_train.shape[1]:>11}{float(d.y_train.sum()):>13.4f}')

print('\n*** Preprocessing check passed — rows, folds, terminals and finiteness all hold ***')

contradiction rows removed [625, 689, 774, 1231, 1242, 1301, 1335]
fold sizes [291, 291, 291, 290, 290] · make_folds agrees with the shared table

 fold  train  valid   test  terminals        y sum
    0  1,162    291  1,459         90   13981.3067
    1  1,162    291  1,459         90   13979.0478
    2  1,162    291  1,459         90   13964.4396
    3  1,163    290  1,459         90   13976.0928
    4  1,163    290  1,459         90   13978.8537

*** Preprocessing check passed — rows, folds, terminals and finiteness all hold ***


# 4. Define the model / 모델 정의

### **모델은 프로그램의 개체군이다**
개체 하나는 터미널 90개와 보호 연산자 8개(`add sub mul div log sqrt max min`)로 만든 DEAP 수식 나무
1~15개의 가변 길이 목록이다. 보호 연산자란 0으로 나누기와 음수 로그가 예외를 던지지 않고 유한한 값을
돌려주도록 감싼 것이다.<br>
<br>
개체를 평가한다는 것은 그 나무들을 학습 행에 대해 열 단위로 계산해 행렬로 쌓고 릿지를 푸는 것이다.
그때의 GCV가 적합도다. 여기에 절약 벌점을 더해 프로그램이 커지는 것만으로 이기지 못하게 한다.<br>
<br>
벌점은 세대가 갈수록 $1/\sqrt{\text{세대}}$로 약해진다. 초반에는 작은 식으로 폭을 넓히고, 후반에는
복잡한 식이 자기 값을 증명할 여지를 준다.

## The material — primitives and the evaluator / 재료 — 연산자와 평가기

### Primitives and the expression evaluator / 연산자와 수식 평가기

In [15]:
# Protected operators, the typed primitive set, and the column-wise expression evaluator.
# 보호 연산자, 원시 집합, 그리고 수식을 열 단위로 계산하는 평가기.
#   protected_div/log/sqrt : 0 나누기·음수 로그에서 예외 대신 유한값을 돌려준다
#   make_pset              : 터미널 90개 + 연산자 8개 + ephemeral 상수
#   Evaluator.forward      : 나무를 후위 순회로 한 번에 계산한다(개체마다 수만 번 불린다)
#   tournament             : 토너먼트 선택(크기 15). 상위 20%가 부모의 96.5%를 공급한다

PROTECT_EPS = 0.001


# Protected operators return a finite value instead of raising, so a candidate never dies of a zero divide.
# 보호 연산자는 예외를 던지지 않고 유한한 값을 돌려준다. 개체가 0 나누기로 죽지 않는다.
# |분모| <= 0.001이면 나눗셈은 1, |x| <= 0.001이면 log는 0, sqrt는 |x|를 쓴다.
def protected_div(a, b):
    if isinstance(b, np.ndarray):
        out = np.ones_like(b)
        np.divide(a, b, out=out, where=np.abs(b) > PROTECT_EPS)
        return out
    if abs(b) > PROTECT_EPS:
        return a / b
    return np.ones_like(a) if isinstance(a, np.ndarray) else np.float64(1.0)

def protected_log(a):
    if isinstance(a, np.ndarray):
        absa = np.abs(a)
        out = np.zeros_like(absa)
        np.log(absa, out=out, where=absa > PROTECT_EPS)
        return out
    absa = abs(a)
    return np.log(absa) if absa > PROTECT_EPS else np.float64(0.0)

def protected_sqrt(a):
    return np.sqrt(np.abs(a))


# 나무의 높이. 상한(6)을 넘는 교차·돌연변이 결과는 버린다.
def tree_height(tree):
    stack = [0]
    pop, push = stack.pop, stack.append
    best = 0
    for node in tree:
        depth = pop()
        if depth > best:
            best = depth
        if node.__class__ is gp.Primitive:
            depth += 1
            arity = node.arity
            if arity == 2:
                push(depth)
                push(depth)
            elif arity == 1:
                push(depth)
            else:
                stack.extend([depth] * arity)
    return best


# Tournament of 15 is the entire selection pressure: draw 15 at random, the lowest fitness wins.
# 토너먼트 15가 선택압 전부다. 무작위로 15개를 뽑아 적합도가 가장 낮은 개체가 이긴다.
# 적합도가 같으면 짧은 개체가 이긴다. 복원 추출이므로 하위 개체도 추첨에 남는다.
def tournament(fitness, length, n, size, rng, entrants=False):
    idx = rng.integers(0, len(fitness), size=(n, size))
    f = fitness[idx]
    tie = np.where(f == f.min(axis=1, keepdims=True), length[idx], np.iinfo(np.int64).max)
    slot = tie.argmin(axis=1)
    winners = idx[np.arange(n), slot]
    return (winners, idx, slot) if entrants else winners

PRIMITIVES = {'add': operator.add, 'sub': operator.sub,
              'mul': operator.mul, 'div': protected_div,
              'log': protected_log, 'sqrt': protected_sqrt,
              'max': np.maximum, 'min': np.minimum}
PRIMITIVES = {k: (v, 1 if k in ('log', 'sqrt') else 2)
              for k, v in PRIMITIVES.items()}

EPHEMERAL = 'c3'

EPHEMERAL_T = 'c4'

N_KEPT = len(KEPT)

ARG_NAMES = list(KEPT) + [f'FAC_{c}' for c in FACTOR_COLUMNS]

K_COL, K_FAC, K_CONST, K_PRIM, K_CAT = 0, 1, 2, 3, 4

class Cat:
    pass

def random_constant():
    return random.uniform(-1.0, 1.0)


# 90 terminals, 8 operators, and an ephemeral constant drawn from U(-1, 1) each time a node is created.
# 터미널 90개, 연산자 8개, 그리고 노드를 만들 때마다 U(-1, 1)에서 뽑는 ephemeral 상수.
def make_pset(names=None):
    names = list(names) if names else ARG_NAMES
    cat = {i for i, n in enumerate(names) if n.startswith('CAT_')}
    if not cat:
        pset = gp.PrimitiveSet('V3', len(names))
        for name, (fn, arity) in PRIMITIVES.items():
            pset.addPrimitive(fn, arity, name=name)
        pset.addEphemeralConstant(EPHEMERAL, random_constant)
    else:
        pset = gp.PrimitiveSetTyped(
            'V4', [Cat if i in cat else float for i in range(len(names))], float)
        for name, (fn, arity) in PRIMITIVES.items():
            pset.addPrimitive(fn, [float] * arity, float, name=name)
        pset.addEphemeralConstant(EPHEMERAL_T, random_constant, float)
    pset.renameArguments(**{f'ARG{i}': n for i, n in enumerate(names)})
    return pset


# 노드 이름 → (종류, 값). 터미널 이름은 renameArguments 뒤에도 ARGi로 남으므로 value를 쓴다.
def node_table(pset, n_cols=None):
    n_cols = N_KEPT if n_cols is None else n_cols
    table = {}
    for terms in pset.terminals.values():
        for term in terms:
            if not isinstance(term, gp.Terminal):
                table[term.name] = (K_CONST, None)
                continue
            i = pset.arguments.index(term.value)
            nm = str(term.value)
            if nm.startswith('CAT_'):
                table[term.name] = (K_CAT, FACTOR_COLUMNS.index(nm[4:]))
            elif nm.startswith('FAC_'):
                table[term.name] = (K_FAC, FACTOR_COLUMNS.index(nm[4:]))
            else:
                table[term.name] = (K_COL, i) if i < n_cols else (K_FAC, i - n_cols)
    for prims in pset.primitives.values():
        for prim in prims:
            table[prim.name] = (K_PRIM, prim.name)
    return table

def is_cat_gene(tree, table):
    return len(tree) == 1 and table.get(tree[0].name, (None,))[0] == K_CAT


# 나무를 (종류, 값) 배열로 바꾼다. forward가 이것을 읽는다.
def compile_tree(tree, table):
    T = len(tree)
    kind = [0] * T
    payload = [None] * T
    kids = [()] * T
    stack = []
    for i in range(T - 1, -1, -1):
        node = tree[i]
        k, v = table.get(node.name, (K_CONST, None))
        if k == K_CONST:
            kind[i], payload[i] = K_CONST, float(node.value)
        elif k == K_PRIM:
            kind[i], payload[i] = K_PRIM, v
            kids[i] = tuple(stack.pop() for _ in range(node.arity))
        else:
            kind[i], payload[i] = k, v
        stack.append(i)
    return kind, payload, kids

def _apply(name, args):
    if name == 'add':
        return args[0] + args[1]
    if name == 'sub':
        return args[0] - args[1]
    if name == 'mul':
        return args[0] * args[1]
    if name == 'div':
        return protected_div(args[0], args[1])
    if name == 'log':
        return protected_log(args[0])
    if name == 'sqrt':
        return protected_sqrt(args[0])
    if name == 'max':
        return np.maximum(args[0], args[1])
    if name == 'min':
        return np.minimum(args[0], args[1])
    raise KeyError(name)

def _partials(name, args):
    if name == 'add':
        return 1.0, 1.0
    if name == 'sub':
        return 1.0, -1.0
    if name == 'mul':
        return args[1], args[0]
    if name == 'div':
        a, b = args
        ok = np.abs(b) > PROTECT_EPS
        inv = np.where(ok, 1.0 / np.where(ok, b, 1.0), 0.0)
        return inv, np.where(ok, -a * inv * inv, 0.0)
    if name == 'log':
        a = args[0]
        ok = np.abs(a) > PROTECT_EPS
        return (np.where(ok, 1.0 / np.where(ok, a, 1.0), 0.0),)
    if name == 'sqrt':
        a = args[0]
        return (np.sign(a) / (2.0 * np.sqrt(np.maximum(np.abs(a), PROTECT_EPS))),)
    if name == 'max':
        a, b = args
        return (a >= b).astype(float), (a < b).astype(float)
    if name == 'min':
        a, b = args
        return (a <= b).astype(float), (a > b).astype(float)
    raise KeyError(name)


# Trees are evaluated column-wise over all rows at once, postfix order, one pass. This is the hot loop.
# 나무를 모든 행에 대해 열 단위로 한 번에 계산한다. 후위 순회 1회다. 가장 뜨거운 루프다.
class Evaluator:

    def __init__(self, X, codes, table):
        self.n = X.shape[0]
        self.cols = [np.ascontiguousarray(X[:, j]) for j in range(X.shape[1])]
        self.codes = [codes[c] for c in FACTOR_COLUMNS if c in (codes or {})]
        self.catcols = [c.astype(float) for c in self.codes]
        self.table = table
        self._ones = np.ones(self.n)
        self._rows = np.arange(self.n)

    def forward(self, tree, theta, keep=False):
        kind, payload, kids = compile_tree(tree, self.table)
        val = [None] * len(kind)
        with np.errstate(all='ignore'):
            for i in range(len(kind) - 1, -1, -1):
                k = kind[i]
                if k == K_COL:
                    val[i] = self.cols[payload[i]]
                elif k == K_FAC:
                    j = payload[i]
                    val[i] = theta[FACTOR_COLUMNS[j]][self.codes[j]]
                elif k == K_CAT:
                    val[i] = self.catcols[payload[i]]
                elif k == K_CONST:
                    val[i] = payload[i] * self._ones
                else:
                    val[i] = _apply(payload[i], [val[c] for c in kids[i]])
        out = val[0]
        out = out if isinstance(out, np.ndarray) and out.shape == (self.n,) else np.broadcast_to(out, (self.n,))
        return (out, (kind, payload, kids, val)) if keep else out

    def backward(self, tape, upstream, sink):
        kind, payload, kids, val = tape
        adj = [None] * len(kind)
        adj[0] = upstream
        with np.errstate(all='ignore'):
            for i in range(len(kind)):
                a = adj[i]
                if a is None:
                    continue
                k = kind[i]
                if k == K_PRIM:
                    ds = _partials(payload[i], [val[c] for c in kids[i]])
                    for c, d in zip(kids[i], ds):
                        contrib = a * d
                        adj[c] = contrib if adj[c] is None else adj[c] + contrib
                elif k == K_FAC:
                    sink(payload[i], a)

    def grad_sink(self, grads):
        def f(j, a):
            np.add.at(grads[FACTOR_COLUMNS[j]], self.codes[j], a)
        return f

    def jac_sink(self, J, base):
        rows = self._rows
        def f(j, a):
            np.add.at(J, (rows, base[j] + self.codes[j]), a)
        return f

## The decider — ridge with GCV / 결정자 — GCV를 쓰는 릿지

### Ridge decider with GCV / GCV로 α를 고르는 릿지 결정자

여기에 더 센 모델이 아니라 릿지가 앉아 있는 이유는 GCV다.<br>
<br>
$$\text{GCV} = \frac{\text{RSS}/n}{(1 - \mathrm{df}/n)^2}, \qquad \mathrm{df} = \operatorname{tr}(H)$$
<br>
해석적 LOO 근사다. 재추출이 없고, 학습 행 $n$개 전부를 쓰고, 개체마다 결정적인 수 하나가 나온다.
두 개체가 같은 근거로 비교된다. α는 개체마다 GCV가 최소인 값을 고른다.<br>
<br>
부트스트랩 OOB 점수는 그 성질이 없다. 나무나 부스터에는 자연스러운 적합도인데도 그렇다. 같은 개체라도
**재추출 표본**이 바뀌면 점수가 흔들리고, 개체 10,000개 중에서 1등을 뽑으면 좋은 개체가 아니라 **운 좋은
표본을 뽑은** 개체가 이긴다. 개체군이 신호가 아니라 추정기의 잡음을 캐게 된다. 같은 예산에서 릿지가
부스터보다 나았던 이유가 이것이다.<br>
<br>
퇴화한 유전자는 `alive` 마스크로 떨어내고 계수를 0으로 둔다.

In [16]:
# The decider: standardize the gene columns, solve ridge over a fixed alpha grid, keep the lowest GCV.
# 결정자: 유전자 열을 표준화하고 고정 α 격자에서 릿지를 풀어 GCV가 가장 낮은 것을 고른다.
#   standardize : 분산이 거의 없는 열을 alive=False로 떨어낸다
#   fit         : α ∈ {0.01, 0.1, 1, 10, 100, 1000} 중 GCV 최소를 고른다
#   gene_matrix : 개체의 나무들을 열로 쌓는다 (n × 유전자수)
#   refine      : theta 경사 정련. 이 노트북에서는 iters=0이라 즉시 반환한다


# α 격자. 개체마다 이 중 GCV가 가장 낮은 값을 고른다.
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)

MIN_STD = 1e-12


# 유전자 열을 표준화한다. 분산이 거의 없는 열은 alive=False로 떨어낸다.
def standardize(G):
    finite = np.isfinite(G).all(axis=0)
    mu = np.where(finite, G.mean(axis=0, where=np.isfinite(G)), 0.0)
    sd = np.where(finite, G.std(axis=0, where=np.isfinite(G)), 0.0)
    alive = finite & (sd > MIN_STD)
    sd_safe = np.where(alive, sd, 1.0)
    Z = (G - mu) / sd_safe
    Z[:, ~alive] = 0.0
    return Z, mu, sd_safe, alive


# GCV = (RSS/n) / (1 - df/n)^2. An analytic leave-one-out estimate: no resampling, all n rows, deterministic.
# GCV는 해석적 LOO 근사다. 재추출이 없고 학습 행 전체를 쓰며 개체마다 결정적인 수 하나가 나온다.
# 그래서 두 개체가 같은 근거로 비교된다. 이것이 적합도다.
def fit(Gmat, y, n_theta=0, alphas=ALPHAS):
    n = len(y)
    Z, mu, sd, alive = standardize(Gmat)
    k = int(alive.sum())
    D = np.empty((n, k + 1))
    D[:, 0] = 1.0
    if k:
        D[:, 1:] = Z[:, alive]
    G = D.T @ D
    c = D.T @ y
    yy = float(y @ y)
    pen = np.eye(k + 1)
    pen[0, 0] = 0.0
    best = None
    for a in alphas:
        A = G + a * pen
        try:
            b = np.linalg.solve(A, c)
            tr = float(np.trace(np.linalg.solve(A, G)))
        except np.linalg.LinAlgError:
            continue
        rss = yy - 2.0 * float(b @ c) + float(b @ G @ b)
        rss = max(rss, 0.0)
        df = tr + n_theta
        denom = 1.0 - df / n
        gcv = np.inf if denom <= 1e-6 else (rss / n) / (denom * denom)
        if not np.isfinite(gcv):
            gcv = np.inf
        if best is None or gcv < best[0]:
            best = (gcv, a, b, rss, df)
    if best is None:
        return None
    gcv, a, b, rss, df = best
    beta = np.zeros(Gmat.shape[1])
    beta[alive] = b[1:]
    return dict(alpha=a, beta=beta, intercept=float(b[0]), mu=mu, sd=sd, alive=alive,
                rss=rss, df=df, gcv_rmse=float(np.sqrt(gcv)), train_rmse=float(np.sqrt(rss / n)))


# 저장된 계수로 다른 행 묶음을 예측한다.
def predict(Gmat, sol):
    if 'est' in sol:
        from . import decider
        return decider.predict(Gmat, sol)
    Z = (Gmat - sol['mu']) / sol['sd']
    Z = np.where(np.isfinite(Z), Z, 0.0)
    Z[:, ~sol['alive']] = 0.0
    return sol['intercept'] + Z @ sol['beta']

def param_layout(n_levels):
    base, p = {}, 0
    for j, c in enumerate(FACTOR_COLUMNS):
        if c not in (n_levels or {}):
            continue
        base[j] = p
        p += n_levels[c]
    return base, p

def normalize(theta, out=None, center=0.0):
    out = out if out is not None else {}
    for c, v in theta.items():
        sd = float(v.std())
        out[c] = (v - v.mean()) / sd + center if sd > 1e-12 else np.full_like(v, center)
    return out


# theta가 쓰는 자유도. 이 노트북에서는 0이다.
def effective_df(n_levels):
    return int(sum(max(n - 2, 0) for n in n_levels.values()))

def flatten(theta):
    parts = [theta[c] for c in FACTOR_COLUMNS if c in theta]
    return np.concatenate(parts) if parts else np.zeros(0)

def unflatten(vec, n_levels):
    out, i = {}, 0
    for c in FACTOR_COLUMNS:
        if c not in (n_levels or {}):
            continue
        n = n_levels[c]
        out[c] = vec[i:i + n].copy()
        i += n
    return out


# The individual's trees become the columns of one matrix. This is what the ridge reads.
# 개체의 나무들이 한 행렬의 열이 된다. 릿지가 읽는 것이 이것이다.
def gene_matrix(ev, genes, theta, keep=False):
    cols, tapes = [], []
    for g in genes:
        if keep:
            v, tp = ev.forward(g, theta, keep=True)
            tapes.append(tp)
        else:
            v = ev.forward(g, theta)
        cols.append(v)
    G = np.column_stack(cols) if cols else np.zeros((ev.n, 0))
    return (G, tapes) if keep else G


# theta 경사 정련. 이 노트북에서는 iters=0이라 즉시 반환한다.
def refine(ev, genes, theta, y, n_levels, iters=10, alpha_theta=0.0, lam=1e-3, n_theta_df=0,
           normalize_theta=True, center=0.0):
    base, p = param_layout(n_levels)
    if normalize_theta:
        theta = normalize(theta, center=center)
    theta0 = flatten(theta)
    cur = theta0.copy()
    curd = unflatten(cur, n_levels)
    G, tapes = gene_matrix(ev, genes, curd, keep=True)
    sol = fit(G, y, n_theta=n_theta_df)
    if sol is None or iters == 0 or p == 0:
        return curd, sol, 0

    def objective(vec):
        td = unflatten(vec, n_levels)
        if normalize_theta:
            td = normalize(td, td, center=center)
        Gm, tp = gene_matrix(ev, genes, td, keep=True)
        s = fit(Gm, y, n_theta=n_theta_df)
        if s is None:
            return np.inf, None, None, None
        r = y - predict(Gm, s)
        d = vec - theta0
        obj = 0.5 * float(r @ r) + 0.5 * alpha_theta * float(d @ d)
        return obj, s, r, tp

    obj, sol, r, tapes = objective(cur)
    accepted = 0
    for _ in range(iters):
        if not np.isfinite(obj):
            break
        J = np.zeros((ev.n, p))
        sink = ev.jac_sink(J, base)
        ones = np.ones(ev.n)
        for k, tp in enumerate(tapes):
            if not sol['alive'][k] or sol['beta'][k] == 0.0:
                continue
            ev.backward(tp, ones * (sol['beta'][k] / sol['sd'][k]), sink)
        if not np.isfinite(J).all():
            J = np.nan_to_num(J, nan=0.0, posinf=0.0, neginf=0.0)
        JtJ = J.T @ J
        g = J.T @ r - alpha_theta * (cur - theta0)
        D = np.diag(np.maximum(np.diag(JtJ), 1e-12))
        step = None
        try:
            step = np.linalg.solve(JtJ + alpha_theta * np.eye(p) + lam * D, g)
        except np.linalg.LinAlgError:
            lam *= 10
            continue
        if not np.isfinite(step).all():
            lam *= 10
            continue
        cand = cur + step
        new_obj, new_sol, new_r, new_tapes = objective(cand)
        if new_obj < obj:
            if normalize_theta:
                cand = flatten(normalize(unflatten(cand, n_levels), center=center))
            cur, obj, sol, r, tapes = cand, new_obj, new_sol, new_r, new_tapes
            lam = max(lam / 10, 1e-12)
            accepted += 1
        else:
            lam *= 10
            if lam > 1e12:
                break
    final = unflatten(cur, n_levels)
    return (normalize(final, center=center) if normalize_theta else final), sol, accepted

## The search — the evolution loop / 탐색 — 진화 루프

### Evolution loop / 진화 루프

반환되는 개체는 마지막 세대의 1등이 아니다. GPTIPS 규칙을 따라 세대마다 train 1등을 그 fold의 검증 행으로
한 번 재고, 실행 전체에서 그 값이 가장 좋았던 개체를 반환한다. 선택압은 검증 행을 보지 않는다. 검증 행은
어느 스냅샷을 남길지만 정한다.<br>

In [17]:
# The evolution loop: evaluate → rank → breed, and the GPTIPS rule for which individual to return.
# 진화 루프. 평가 → 순위 → 번식을 돌고, 반환할 개체는 GPTIPS 규칙으로 고른다.
#   assign_fitness        : GCV + 공변 유도 절약 벌점(÷√세대). 벌점 계수는 세대마다 다시 계산한다
#   run                   : 세대 루프. GPTIPS 규칙으로 반환 개체를 고른다
#   cfg['max_seconds']    : 벽시계 상한. Kaggle 세션 한도 안에서 깔끔히 멈춘다


# 기본 설정. 노트북의 CFG가 필요한 값을 덮어쓴다.
DEFAULTS = dict(population=10000, genes=15, variable_genes=True, max_height=6, init_height=(2, 6), mut_height=(0, 2),
                parsimony=1e-4, tournament=15, p_crossover=0.7, p_cross_hi=0.2,
                alpha_theta=0.0, lm_early=10, lm_early_gens=10, lm_late=1, max_gen=120, seed=42,
                normalize_theta=True, theta_center='auto',
                covariant_parsimony=True, parsimony_scale=1.0, size_measure='linear',
                parsimony_decay='sqrt_gen', parsimony_target=None,
                decider='ridge', tree_depth=6,
                es_patience=0, es_min_gens=10, max_seconds=0,
                ckpt_top=0,
                gbm_iters=50, gbm_depth=3, gbm_lr=0.1, gbm_bins=32)


# 개체 하나. 유전자 목록과 릿지 해, 그리고 계보(uid·parents·op)를 들고 있다.
class Ind:
    __slots__ = ('genes', 'theta', 'fit', 'sol', 'nodes', 'n_theta', 'uid', 'parents', 'op')


    def __init__(self, genes, theta):
        self.genes, self.theta = genes, theta
        self.fit = np.inf
        self.sol = None
        self.nodes = sum(len(g) for g in genes)
        self.n_theta = sum(len(v) for v in theta.values())
        self.uid, self.parents, self.op = 0, (), 'init'

    def copy(self):
        c = Ind(list(self.genes), {k: v.copy() for k, v in self.theta.items()})
        c.fit, c.sol = self.fit, self.sol
        c.uid, c.parents, c.op = self.uid, self.parents, self.op
        return c


# 같은 식을 찾아내는 열쇠. 한 개체 안의 중복 유전자를 줄이는 데 쓴다.
def gene_key(tree):
    return tuple(n.format() if n.arity == 0 else n.name for n in tree)

def _new_gene(pset, cfg):
    if cfg.get('p_cat_gene', 0.0) and random.random() < cfg['p_cat_gene']:
        return gp.PrimitiveTree([random.choice(pset.terminals[Cat])])
    return gp.PrimitiveTree(gp.genHalfAndHalf(pset, cfg['init_height'][0], cfg['init_height'][1]))


# 한 개체 안에서 같은 식이 둘 이상이면 하나로 줄인다.
def dedup(ind, pset, cfg):
    seen, out = set(), []
    for g in ind.genes:
        k = gene_key(g)
        if k in seen:
            if cfg['variable_genes']:
                continue
            g = _new_gene(pset, cfg)
            k = gene_key(g)
        seen.add(k)
        out.append(g)
    if not out:
        out = [_new_gene(pset, cfg)]
    ind.genes = out
    ind.nodes = sum(len(g) for g in out)
    return len(ind.genes)

def evaluate(ind, ev, y, n_levels, cfg, iters, bag_seed=0, ev_va=None, yv=None):
    df = effective_df(n_levels) if cfg['normalize_theta'] else ind.n_theta
    theta, sol, _ = refine(ev, ind.genes, ind.theta, y, n_levels, iters=iters,
                              alpha_theta=cfg['alpha_theta'], n_theta_df=df,
                              normalize_theta=cfg['normalize_theta'], center=cfg['theta_center'])
    if sol is None:
        ind.fit, ind.sol = np.inf, None
        return
    ind.theta, ind.sol = theta, sol


# 개체군 전부의 부모-자식 간선. 계보 그래프의 재료다.
def _lineage(pop, gen):
    out = []
    for i in pop:
        g = i.sol['gcv_rmse'] if i.sol is not None else float('inf')
        out.append((gen, i.uid,
                    i.parents[0] if i.parents else -1,
                    i.parents[1] if len(i.parents) > 1 else -1,
                    i.op, i.fit, g, i.nodes, len(i.genes)))
    return out


# 세대마다 어떤 터미널·연산자·식이 얼마나 쓰였는지 센다. 잔존률 그림의 재료다.
def _census(pop, top_n=20):
    terms, prims, exprs = Counter(), Counter(), Counter()
    n = 0
    for ind in pop:
        if ind.sol is None:
            continue
        for g in ind.genes:
            n += 1
            exprs[str(g)] += 1
            for node in g:
                if isinstance(node, gp.Primitive):
                    prims[node.name] += 1
                elif isinstance(getattr(node, 'value', None), str):
                    terms[node.value] += 1
                else:
                    terms['상수'] += 1
    n_ind = sum(1 for i in pop if i.sol is not None)
    return dict(terminals=terms, prims=prims, exprs=exprs.most_common(top_n),
                n_genes=n, n_ind=n_ind)


# 개체군 수준 통계. 1등이 벌점에 묶여 얼어도 pop_min_gcv는 움직인다.
def _pop_stats(pop, best):
    sols = [i.sol for i in pop if i.sol is not None]
    if not sols:
        return dict(pop_min_gcv=np.inf, avg_trees=np.nan, cat_share=np.nan,
                    avg_cat=np.nan, uniq_gcv=0, best_trees=np.nan, best_cat=np.nan,
                    **{f'gcv_q{k:02d}': np.nan for k in (5, 25, 50, 75, 95)})
    gcv = np.array([s['gcv_rmse'] for s in sols], float)
    trees = [s.get('trees') for s in sols if s.get('trees') is not None]
    ncat = [s.get('n_cat') for s in sols if s.get('n_cat') is not None]
    ngen = [len(i.genes) for i in pop if i.sol is not None]
    bs = best.sol or {}
    q = np.percentile(gcv[np.isfinite(gcv)], [5, 25, 50, 75, 95]) if np.isfinite(gcv).any() \
        else [np.nan] * 5
    return dict(
        gcv_q05=float(q[0]), gcv_q25=float(q[1]), gcv_q50=float(q[2]),
        gcv_q75=float(q[3]), gcv_q95=float(q[4]),
        pop_min_gcv=float(gcv.min()),
        uniq_gcv=int(len(np.unique(np.round(gcv, 9)))),
        avg_trees=(float(np.mean(trees)) if trees else np.nan),
        avg_cat=(float(np.mean(ncat)) if ncat else np.nan),
        cat_share=(float(np.sum(ncat) / np.sum(ngen)) if ncat and np.sum(ngen) else np.nan),
        best_trees=float(bs.get('trees', np.nan)),
        best_cat=float(bs.get('n_cat', np.nan)))

def _size(nodes, measure):
    return np.sqrt(nodes) if measure == 'sqrt' else nodes


# Covariant parsimony (Poli & McPhee, GECCO 2008): the penalty coefficient is refitted every generation.
# 공변 절약 벌점. 벌점 계수를 세대마다 다시 맞춘다. 1/sqrt(세대)로 약해져 후반에 복잡한 식을 허용한다.
def parsimony_coefficient(pop, scale=1.0, measure='linear'):
    e = np.array([i.sol['gcv_rmse'] for i in pop if i.sol is not None], float)
    l = _size(np.array([i.nodes for i in pop if i.sol is not None], float), measure)
    if len(e) < 10:
        return 0.0
    var = float(np.var(l))
    if var <= 0 or not np.isfinite(var):
        return 0.0
    c = -float(np.cov(l, e)[0, 1]) / var
    return max(0.0, c) * scale


# Fitness = GCV + parsimony penalty. A program cannot win by growing.
# 적합도 = GCV + 절약 벌점. 프로그램이 커지는 것만으로는 이기지 못한다.
def assign_fitness(pop, df, cfg, gen=0, state=None):
    m = cfg['size_measure']
    if not cfg['covariant_parsimony']:
        c = cfg['parsimony']
    else:
        raw = parsimony_coefficient(pop, 1.0, m)
        mode = cfg['parsimony_decay']
        if mode == 'sqrt_gen':
            c = raw * cfg['parsimony_scale'] / np.sqrt(max(gen, 1))
        elif mode == 'target' and cfg['parsimony_target']:
            st = state if state is not None else {}
            mean_size = float(np.mean([i.nodes for i in pop]))
            g = st.get('gain', 1.0)
            g = float(np.clip(g * np.clip(mean_size / cfg['parsimony_target'], 1 / 1.3, 1.3), 1e-3, 1e3))
            st['gain'] = g
            c = raw * g * cfg['parsimony_scale']
        else:
            c = raw * cfg['parsimony_scale']
    for ind in pop:
        ind.fit = (np.inf if ind.sol is None
                   else ind.sol['gcv_rmse'] + c * (_size(ind.nodes, m) + df))
    return c


# 개체의 식과 릿지 해로 다른 행 묶음을 예측한다.
def predict_on(ind, ev):
    if ind.sol is None:
        return None
    pred = predict(gene_matrix(ev, ind.genes, ind.theta), ind.sol)
    return pred if np.isfinite(pred).all() else None

def rmse(pred, y):
    if pred is None:
        return np.inf
    d = pred - y
    return float(np.sqrt(d @ d / len(d)))


# 세대마다 train 1등을 검증 행으로 한 번 재는 데 쓴다.
def valid_rmse(ind, ev_valid, y_valid):
    return rmse(predict_on(ind, ev_valid), y_valid)


# The GPTIPS rule: selection pressure comes from training fitness only. The fold's validation rows are scored
# once per generation on the training champion, and the run returns the best of those snapshots.
# GPTIPS 규칙. 선택압은 train 적합도에서만 나온다. fold의 검증 행은 세대마다 train 1등을 한 번 재는 데만
# 쓰이고, 실행은 그중 가장 좋았던 스냅샷을 반환한다. 검증 행은 선택에 끼어들지 않는다.
def run(design, cfg=None, log=None, ckpt=None):
    cfg = dict(DEFAULTS, **(cfg or {}))
    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    rng = np.random.default_rng(cfg['seed'])
    pset = make_pset(getattr(design, 'arg_names', None))
    table = node_table(pset, getattr(design, 'n_cols', None))
    ev = Evaluator(design.X_train, design.codes_train, table)
    ev_va = (Evaluator(design.X_valid, design.codes_valid, table)
             if design.X_valid is not None else None)
    y, yv = design.y_train, design.y_valid
    N, K = int(cfg['population']), int(cfg['genes'])
    if cfg['theta_center'] == 'auto':
        cfg['theta_center'] = float(np.median(design.X_train.mean(axis=0)))
    mate = gp.cxOnePoint

    def mut_expr(pset, type_=None):
        if type_ is not None and not pset.primitives.get(type_):
            return [random.choice(pset.terminals[type_])]
        return gp.genFull(pset, cfg['mut_height'][0], cfg['mut_height'][1], type_)

    n_cat = len(pset.terminals.get(Cat, ()))
    cfg.setdefault('p_cat_gene', n_cat / len(pset.arguments) if n_cat else 0.0)
    t0 = time.perf_counter()

    def fresh():
        n = random.randint(1, K) if cfg['variable_genes'] else K
        genes = [_new_gene(pset, cfg) for _ in range(n)]
        theta = {c: v + rng.normal(0, 0.1, size=v.shape) for c, v in design.theta_init.items()}
        if cfg['normalize_theta']:
            theta = normalize(theta, center=cfg['theta_center'])
        return Ind(genes, theta)

    nuid = [0]

    def _tag(ind, parents, op):
        nuid[0] += 1
        ind.uid, ind.parents, ind.op = nuid[0], tuple(parents), op
        return ind

    pop = [_tag(fresh(), (), 'init') for _ in range(N)]
    for ind in pop:
        dedup(ind, pset, cfg)
    rows = []
    pstate = {}
    stop = 'max_gen'
    nbag = [0]
    val_best = None
    for gen in range(int(cfg['max_gen']) + 1):
        iters = cfg['lm_early'] if gen < cfg['lm_early_gens'] else cfg['lm_late']
        for ind in pop:
            if ind.sol is None:
                nbag[0] += 1
                evaluate(ind, ev, y, design.n_levels, cfg, iters,
                         bag_seed=nbag[0] + 7919 * cfg['seed'], ev_va=ev_va, yv=yv)
        c_used = assign_fitness(pop, effective_df(design.n_levels), cfg, gen, pstate)
        fit = np.array([i.fit for i in pop])
        length = np.array([i.nodes for i in pop], dtype=np.int64)
        b = int(np.lexsort((length, fit))[0])
        best = pop[b]
        row = dict(gen=gen, best_fit=best.fit, best_train=best.sol['train_rmse'] if best.sol else np.inf,
                   best_gcv=best.sol['gcv_rmse'] if best.sol else np.inf,
                   best_valid=valid_rmse(best, ev_va, yv) if ev_va is not None else np.nan,
                   best_nodes=best.nodes, best_alive=int(best.sol['alive'].sum()) if best.sol else 0,
                   best_genes=len(best.genes), avg_genes=float(np.mean([len(i.genes) for i in pop])),
                   best_alpha=best.sol['alpha'] if best.sol else np.nan,
                   avg_fit=float(np.mean(fit[np.isfinite(fit)])) if np.isfinite(fit).any() else np.inf,
                   parsimony_c=c_used, avg_nodes=float(np.mean([i.nodes for i in pop])),
                   n_inf=int((~np.isfinite(fit)).sum()), lm_iters=iters,
                   elapsed=time.perf_counter() - t0,
                   **_pop_stats(pop, best))
        if ev_va is not None and np.isfinite(row['best_valid']):
            if val_best is None or row['best_valid'] < val_best[0]:
                val_best = (row['best_valid'], gen, best.copy())
        row['val_best_so_far'] = val_best[0] if val_best else np.nan
        rows.append(row)
        if ckpt and int(cfg.get('ckpt_top', 0)):
            order = np.lexsort((length, fit))[:int(cfg['ckpt_top'])]
            top, vs = [], []
            for rank, j in enumerate(order):
                ind = pop[int(j)]
                if ind.sol is None:
                    continue
                if rank == 0:
                    v = row['best_valid']
                else:
                    v = valid_rmse(ind, ev_va, yv) if ev_va is not None else np.nan
                top.append(ind)
                vs.append(v)
            ckpt(gen, top, vs, val_best, dict(
                census=(_census(pop, int(cfg.get('census_top', 20)))
                        if cfg.get('census_top', 20) else None),
                lineage=(_lineage(pop, gen) if cfg.get('lineage', 1) else None)))
        if log:
            log(row)
        if gen == int(cfg['max_gen']):
            break
        if cfg.get('max_seconds') and (time.perf_counter() - t0) > float(cfg['max_seconds']):
            stop = 'time'
            break
        if (cfg.get('es_patience') and val_best is not None
                and gen >= int(cfg.get('es_min_gens', 0))
                and gen - val_best[1] >= int(cfg['es_patience'])):
            stop = 'patience'
            break
        elite = _tag(best.copy(), (best.uid,), 'elite')
        n_child = N - 1
        n_pair = (n_child + 1) // 2
        parents = tournament(fit, length, 2 * n_pair, int(cfg['tournament']), rng)
        children = []
        for a, c in zip(parents[0::2], parents[1::2]):
            p1, p2 = pop[a].copy(), pop[c].copy()
            op_of = ['?', '?']
            if random.random() < cfg['p_crossover']:
                if random.random() < cfg['p_cross_hi']:
                    op_of = ['xhi', 'xhi']
                    if cfg['variable_genes']:
                        i1 = random.sample(range(len(p1.genes)), random.randint(1, len(p1.genes)))
                        i2 = random.sample(range(len(p2.genes)), random.randint(1, len(p2.genes)))
                        take1 = [p2.genes[i] for i in i2]
                        take2 = [p1.genes[i] for i in i1]
                        keep1 = [g for j, g in enumerate(p1.genes) if j not in set(i1)]
                        keep2 = [g for j, g in enumerate(p2.genes) if j not in set(i2)]
                        p1.genes = (keep1 + take1)[:K] or [_new_gene(pset, cfg)]
                        p2.genes = (keep2 + take2)[:K] or [_new_gene(pset, cfg)]
                    else:
                        m = random.randint(1, max(1, K // 2))
                        for i in random.sample(range(K), m):
                            p1.genes[i], p2.genes[i] = p2.genes[i], p1.genes[i]
                else:
                    op_of = ['xlo', 'xlo']
                    i1 = random.randrange(len(p1.genes))
                    i2 = random.randrange(len(p2.genes))
                    g1, g2 = mate(gp.PrimitiveTree(p1.genes[i1]), gp.PrimitiveTree(p2.genes[i2]))
                    if tree_height(g1) <= cfg['max_height']:
                        p1.genes[i1] = g1
                    if tree_height(g2) <= cfg['max_height']:
                        p2.genes[i2] = g2
            else:
                for _k, p in enumerate((p1, p2)):
                    u = random.random()
                    if cfg['variable_genes'] and u < 0.1 and len(p.genes) < K:
                        op_of[_k] = 'gene+'
                        p.genes = p.genes + [_new_gene(pset, cfg)]
                    elif cfg['variable_genes'] and u < 0.2 and len(p.genes) > 1:
                        op_of[_k] = 'gene-'
                        p.genes = p.genes[:random.randrange(len(p.genes))] + \
                                  p.genes[random.randrange(len(p.genes)) + 1:] or [p.genes[0]]
                        if not p.genes:
                            p.genes = [_new_gene(pset, cfg)]
                    elif cfg.get('p_cat_gene') and is_cat_gene(
                            p.genes[(i := random.randrange(len(p.genes)))], table) and random.random() < 0.5:
                        op_of[_k] = 'mut_cat'
                        p.genes[i] = _new_gene(pset, cfg)
                    else:
                        op_of[_k] = 'mut_sub'
                        i = random.randrange(len(p.genes))
                        g, = gp.mutUniform(gp.PrimitiveTree(p.genes[i]), expr=mut_expr, pset=pset)
                        if tree_height(g) <= cfg['max_height']:
                            p.genes[i] = g
            for p, o in zip((p1, p2), op_of):
                p.nodes = sum(len(g) for g in p.genes)
                dedup(p, pset, cfg)
                p.sol = None
                _tag(p, (pop[a].uid, pop[c].uid), o)
            children += [p1, p2]
        pop = [elite] + children[:n_child]
    return dict(rows=rows, best=pop[b] if b < len(pop) else pop[0], pop=pop, stop=stop,
                val_best=val_best, last_valid=rows[-1]['best_valid'],
                runtime=time.perf_counter() - t0)

### Ridge-90 baseline / 릿지 90열 기준선

In [18]:
# A plain ridge on all 90 terminals, fold by fold. This is the bar the GP has to clear, and the
# score gate in section 8 compares against it — same rows, same split, so it is a fair ruler.
# 터미널 90개를 그대로 릿지에 넣은 값이다. GP가 넘어야 하는 기준선이고, 8절의 점수 문턱이 이것과
# 비교한다. 같은 행·같은 분할에서 재므로 분할 난이도가 달라져도 자로서 흔들리지 않는다.
RIDGE90 = {}
print(f'{"fold":>5}{"alpha":>9}{"valid":>9}')
for f in range(5):
    d = DES[f]
    _sol = fit(d.X_train, d.y_train)
    RIDGE90[f] = float(np.sqrt(np.mean((predict(d.X_valid, _sol) - d.y_valid) ** 2)))
    print(f'{f:>5}{_sol["alpha"]:>9}{RIDGE90[f]:>9.4f}')

print(f'\nridge-90 mean valid {np.mean(list(RIDGE90.values())):.4f} — the GP has to beat this')

 fold    alpha    valid
    0     10.0   0.1380
    1    100.0   0.1879
    2     10.0   0.1513
    3     10.0   0.1244
    4     10.0   0.1198

ridge-90 mean valid 0.1443 — the GP has to beat this


# 5. Train / 학습

fold 5개를 코어 하나당 워커 하나로 돌린다. 워커는 `fork`로 설계행렬을 물려받으므로 큰 것이 피클을 타고
오가지 않는다. 돌아오는 것은 fold별 결과뿐이다.<br>
<br>
본 런은 fold당 개체군 10,000 × 300세대고, 코어 4개에서 fold당 20분쯤이라 두 물결로 약 40분이다. 세션이
모자라면 `cfg['max_seconds']`가 fold를 깔끔히 멈추고 그 시점의 최고 개체를 반환한다.<br>
<br>
메모리는 개체당 수십 KB 수준이라 개체군 10,000에서도 워커 하나가 수백 MB를 넘지 않는다.

In [19]:
# One worker per fold. fork inherits DES/CFG, so only the result is pickled back.
# fold마다 워커 하나. fork가 DES·CFG를 물려주므로 결과만 피클로 돌아온다.
import multiprocessing as mp
from collections import Counter

VIZ = OUTPUT_DIR / 'viz'          # Kaggle output — downloadable / 캐글 출력, 내려받을 수 있다
VIZ.mkdir(parents=True, exist_ok=True)
OPS = ['init', 'elite', 'xhi', 'xlo', 'mut_sub', 'mut_cat', 'gene+', 'gene-', '?']


def run_fold(f):
    _t = time.time()
    d = DES[f]
    top_rows, cen_rows, exp_rows, lin = [], [], [], []

    def on_gen(gen, top, vs, vb, extra):
        '''Called at the end of every generation — this is where the figures' data comes from.
        세대가 끝날 때마다 불린다. 그림의 재료가 전부 여기서 나온다.'''
        for rank, (ind, v) in enumerate(zip(top, vs), 1):
            s = ind.sol or {}
            top_rows.append(dict(
                gen=gen, rank=rank, uid=ind.uid,
                p1=(ind.parents[0] if ind.parents else -1),
                p2=(ind.parents[1] if len(ind.parents) > 1 else -1), op=ind.op,
                gcv=s.get('gcv_rmse', np.nan), train=s.get('train_rmse', np.nan), valid=v,
                nodes=ind.nodes, n_genes=len(ind.genes),
                alive=int(s['alive'].sum()) if s else 0, alpha=s.get('alpha', np.nan),
                expr=' | '.join(str(g) for g in ind.genes)))
        cen = (extra or {}).get('census')
        if cen:
            n_ind = max(1, cen['n_ind'])
            for kind, cnt in (('terminal', cen['terminals']), ('prim', cen['prims'])):
                tot = max(1, sum(cnt.values()))
                for name, c in cnt.most_common():
                    cen_rows.append(dict(gen=gen, kind=kind, name=name, count=c,
                                         share=c / tot, per_ind=c / n_ind))
            for r, (e, c) in enumerate(cen['exprs'], 1):
                exp_rows.append(dict(gen=gen, rank=r, count=c,
                                     share=c / max(1, cen['n_genes']), expr=e))
        if (extra or {}).get('lineage'):
            lin.extend(extra['lineage'])

    res = run(d, dict(CFG), ckpt=on_gen)
    vb = res['val_best']
    ind = vb[2] if vb is not None else res['best']
    pset = make_pset(d.arg_names)
    table = node_table(pset, d.n_cols)
    ev_va = Evaluator(d.X_valid, d.codes_valid or {}, table)
    ev_te = Evaluator(d.X_test, d.codes_test or {}, table)
    p_va = predict(gene_matrix(ev_va, ind.genes, ind.theta), ind.sol)
    p_te = predict(gene_matrix(ev_te, ind.genes, ind.theta), ind.sol)

    # Write the records as Kaggle output / 기록을 캐글 출력으로 저장한다
    pd.DataFrame(res['rows']).to_csv(VIZ / f'generations_fold{f}.csv', index=False)
    pd.DataFrame(top_rows).to_csv(VIZ / f'top_fold{f}.csv', index=False)
    pd.DataFrame(cen_rows).to_csv(VIZ / f'census_fold{f}.csv', index=False)
    pd.DataFrame(exp_rows).to_csv(VIZ / f'genes_fold{f}.csv', index=False)
    if lin:
        # 개체군 전부의 간선. 압축 npz로 둔다(본 런은 fold당 300만 행이다)
        _code = {o: i for i, o in enumerate(OPS)}
        np.savez_compressed(
            VIZ / f'lineage_fold{f}.npz',
            gen=np.array([t[0] for t in lin], np.int32),
            uid=np.array([t[1] for t in lin], np.int32),
            p1=np.array([t[2] for t in lin], np.int32),
            p2=np.array([t[3] for t in lin], np.int32),
            op=np.array([_code.get(t[4], len(OPS) - 1) for t in lin], np.uint8),
            fit=np.array([t[5] for t in lin], np.float32),
            gcv=np.array([t[6] for t in lin], np.float32),
            nodes=np.array([t[7] for t in lin], np.int32),
            n_genes=np.array([t[8] for t in lin], np.int32),
            ops=np.array(OPS))
    return dict(fold=f, gen=int(vb[1]) if vb is not None else int(CFG['max_gen']),
                valid=float(vb[0]) if vb is not None else float(res['last_valid']),
                stop=res.get('stop'), n_gen=len(res['rows']), uid=int(ind.uid),
                genes=[str(g) for g in ind.genes], nodes=int(ind.nodes),
                alpha=float(ind.sol['alpha']), alive=int(ind.sol['alive'].sum()),
                n_lineage=len(lin), pred_valid=p_va.tolist(), pred_test=p_te.tolist(),
                y_valid=d.y_valid.tolist(), ids_valid=d.ids_valid.tolist(),
                ids_test=d.ids_test.tolist(), secs=time.time() - _t, rss=RSS_MB())


_t0 = time.time()
RES = {}
with ProcessPoolExecutor(max_workers=PROCS, mp_context=mp.get_context('fork')) as ex:
    for r in ex.map(run_fold, range(5)):
        RES[r['fold']] = r
        print(f'  fold {r["fold"]}: gen {r["gen"]:>3} · valid {r["valid"]:.4f} '
              f'· genes {len(r["genes"])}/{r["alive"]} · nodes {r["nodes"]:>3} '
              f'· stop {r["stop"]} · lineage {r["n_lineage"]:,} edges '
              f'· {r["secs"]/60:.1f} min · {r["rss"]:.0f} MB', flush=True)

print(f'\n*** {(time.time()-_t0)/60:.1f} min · mean valid '
      f'{np.mean([RES[f]["valid"] for f in range(5)]):.4f} ***')
print(f'records -> {VIZ}  ({sum(1 for _ in VIZ.iterdir())} files, '
      f'{sum(p.stat().st_size for p in VIZ.iterdir())/1e6:.1f} MB)')

  fold 0: gen  71 · valid 0.1392 · genes 15/15 · nodes  32 · stop max_gen · lineage 3,010,000 edges · 125.4 min · 973 MB
  fold 1: gen  19 · valid 0.1488 · genes 11/11 · nodes  43 · stop max_gen · lineage 3,010,000 edges · 133.9 min · 975 MB
  fold 2: gen  65 · valid 0.1261 · genes 15/15 · nodes  29 · stop max_gen · lineage 3,010,000 edges · 123.1 min · 973 MB
  fold 3: gen  21 · valid 0.1207 · genes 9/9 · nodes  52 · stop max_gen · lineage 3,010,000 edges · 135.2 min · 972 MB
  fold 4: gen  21 · valid 0.1061 · genes 10/10 · nodes  47 · stop max_gen · lineage 3,010,000 edges · 83.5 min · 975 MB

*** 206.6 min · mean valid 0.1282 ***
records -> /kaggle/working/outputs/viz  (25 files, 155.3 MB)


# 6. Validation / 검증

보는 것은 두 가지다. 반환된 예측을 반환된 유전자로 다시 계산해 일치하는지 본다. 결과가 온전히 돌아오지
않은 경우가 여기서 걸린다. 그리고 제출 계약을 본다. 행수, `sample_submission`과의 Id 순서, 유한성, 가격 범위다.<br>
<br>
fold valid 점수를 리더보드 추정값으로 내놓지는 않는다. 검증 행은 각 fold가 반환할 개체를 고르는 데 썼으니
더 이상 편향 없는 추정값이 아니다. 정직한 심판은 리더보드다. 다만 그 점수가 기록에서 크게 벗어나면
제출을 막는 문턱으로는 쓴다. 8절의 관문이 그 일을 한다.

In [20]:
# Check 2 of 4 — do the returned genes reproduce the returned predictions?
# 검증 2/4 — 반환된 유전자가 반환된 예측을 재현하는가?
# The training log-price range is the yardstick for "is this prediction plausible".
# 학습 log 가격 범위가 '이 예측이 그럴듯한가'의 자다.
Y_LO = float(min(DES[f].y_train.min() for f in range(5)))
Y_HI = float(max(DES[f].y_train.max() for f in range(5)))
print(f'training log-price range [{Y_LO:.3f}, {Y_HI:.3f}] '
      f'= ${np.exp(Y_LO):,.0f} ~ ${np.exp(Y_HI):,.0f}\n')

sub_head = pd.read_csv(COMP_DIR / 'sample_submission.csv', nrows=0)
assert list(sub_head.columns) == ['Id', 'SalePrice'], 'sample_submission columns changed!!'
sample_ids = pd.read_csv(COMP_DIR / 'sample_submission.csv').Id.to_numpy()

for f in range(5):
    r, d = RES[f], DES[f]
    pset = make_pset(d.arg_names)
    table = node_table(pset, d.n_cols)
    genes = [gp.PrimitiveTree.from_string(s, pset) for s in r['genes']]
    theta = {k: np.asarray(v, float) for k, v in (d.theta_init or {}).items()}
    ev = Evaluator(d.X_valid, d.codes_valid or {}, table)
    # 저장된 계수로 다시 적합하지 않고, 유전자로 만든 행렬에 릿지를 다시 풀어 valid를 재계산한다
    G = gene_matrix(Evaluator(d.X_train, d.codes_train or {}, table), genes, theta)
    sol = fit(G, d.y_train)
    rv = float(np.sqrt(np.mean((predict(gene_matrix(ev, genes, theta), sol) - d.y_valid) ** 2)))

    pv = np.array(r['pred_valid'])
    assert np.isfinite(pv).all(), f'fold {f} valid prediction has non-finite values!!'
    assert abs(float(np.sqrt(np.mean((pv - np.array(r['y_valid'])) ** 2))) - r['valid']) < 1e-9, \
        f'fold {f} returned valid rmse does not match its own predictions!!'
    assert abs(rv - r['valid']) < 1e-6, \
        f'fold {f} refit from genes gives {rv:.6f}, returned {r["valid"]:.6f}!!'

    # The verified model goes to the output too — genes and their ridge weights are what the figures read.
    # 검증을 통과한 모델도 출력에 남긴다. 그림이 읽는 것은 유전자와 그 릿지 계수다.
    (VIZ / f'model_fold{f}.json').write_text(json.dumps(dict(
        fold=f, gen=int(r['gen']), uid=int(r['uid']), valid=float(r['valid']), n_gen=int(r['n_gen']),
        alpha=float(sol['alpha']), intercept=float(sol['intercept']),
        beta=[float(b) for b in sol['beta']], alive=[bool(a) for a in sol['alive']],
        genes=list(r['genes']), nodes=int(r['nodes'])), indent=1), encoding='utf-8')

    pt = np.array(r['pred_test'])
    assert len(pt) == len(sample_ids) and np.isfinite(pt).all(), f'fold {f} test prediction!!'
    assert list(r['ids_test']) == sample_ids.tolist(), f'fold {f} test Id order != sample_submission!!'
    # A single fold may extrapolate wildly on one test row; the five-fold mean absorbs it. So the hard
    # guard is on the median, and rows outside the training price range are reported, not fatal.
    # fold 하나가 test 한 행에서 크게 외삽할 수 있고, 그건 fold 5개 평균이 흡수한다. 그래서 단단한 검사는
    # 중앙값에만 걸고, 학습 가격 범위를 벗어난 행은 세어서 알리기만 한다.
    _out = int(((pt < Y_LO) | (pt > Y_HI)).sum())
    assert Y_LO < float(np.median(pt)) < Y_HI, \
        f'fold {f} median prediction {np.median(pt):.2f} is outside the training price range!!'
    print(f'  fold {f}: valid {r["valid"]:.4f} · refit from genes {rv:.4f} '
          f'· test range [{pt.min():.2f}, {pt.max():.2f}]'
          f'{f" · {_out} row(s) outside training range" if _out else ""} · {r["n_gen"]} generations logged')

print('\n*** Validation passed — genes reproduce their predictions, Id order matches ***')

training log-price range [10.460, 13.534] = $34,900 ~ $755,000

  fold 0: valid 0.1392 · refit from genes 0.1392 · test range [10.85, 13.24] · 301 generations logged
  fold 1: valid 0.1488 · refit from genes 0.1488 · test range [10.77, 13.39] · 301 generations logged
  fold 2: valid 0.1261 · refit from genes 0.1261 · test range [8.92, 13.24] · 1 row(s) outside training range · 301 generations logged
  fold 3: valid 0.1207 · refit from genes 0.1207 · test range [10.84, 13.23] · 301 generations logged
  fold 4: valid 0.1061 · refit from genes 0.1061 · test range [10.87, 13.19] · 301 generations logged

*** Validation passed — genes reproduce their predictions, Id order matches ***


# 7. Records for the figures / 그림을 위한 기록

그림의 재료는 학습 중에 세대마다 쌓인다. fold마다 파일 여섯 개가 `outputs/viz/`에 떨어지고, 이 절은
그것이 다 떨어졌는지만 본다. 그림은 내려받아 로컬에서 그린다.<br>
<br>
| 파일 | 내용 | 쓰임 |
| --- | --- | --- |
| `generations_fold{f}.csv` | 세대별 1등 지표와 개체군 통계 | 궤적, 분포 부채꼴 |
| `top_fold{f}.csv` | 세대마다 상위 5개의 `uid`·부모·연산자·유전자식 | 세대별 1등, 유전자 유지/버림/새로 |
| `lineage_fold{f}.npz` | 개체군 전부의 부모-자식 간선 | 계보 그래프 |
| `census_fold{f}.csv` | 터미널과 연산자의 세대별 점유율 | 유전자 잔존률 |
| `genes_fold{f}.csv` | 세대마다 가장 흔한 유전자식 50개 | 식 단위 유행과 소멸 |
| `model_fold{f}.json` | 반환 모델 — 유전자와 릿지 계수·α (6절이 검증 뒤 저장) | 유전자 영향력 |
<br>
계보는 본 런에서 fold당 수백만 간선이라 압축 npz로 두고 나머지는 CSV다. 점검이 끝나면 30개를
`viz_records.zip` 하나로 묶는다. Kaggle의 제출 파일 목록이 출력 루트를 알파벳 순으로 늘어놓아서,
느슨한 파일이 많으면 `submission.csv`가 맨 끝으로 밀리기 때문이다.

In [21]:
# The figures are drawn locally from the download. All this cell does is confirm nothing is missing.
# 그림은 내려받아 로컬에서 그린다. 이 셀은 빠진 파일이 없는지만 확인한다.
_want = ('generations', 'top', 'census', 'genes', 'model') + (('lineage',) if CFG['lineage'] else ())
_ext = {'lineage': 'npz', 'model': 'json'}

print(f'{"fold":>5}' + ''.join(f'{_n:>14}' for _n in _want))
_total = 0
for f in range(5):
    _cells = []
    for _n in _want:
        _p = VIZ / f'{_n}_fold{f}.{_ext.get(_n, "csv")}'
        assert _p.exists(), f'{_p.name} is missing — the records are incomplete!!'
        assert _p.stat().st_size > 0, f'{_p.name} is empty!!'
        _total += _p.stat().st_size
        _cells.append(f'{_p.stat().st_size / 1024:>10,.0f} KB')
    print(f'{f:>5}' + ''.join(f'{_c:>14}' for _c in _cells))

# The population statistics and the lineage columns are what the figures need — a wrong CFG would
# write the files anyway, just without them.
# 개체군 통계와 계보 열이 그림에 필요한 것이다. CFG가 틀리면 파일은 그대로 쓰이고 그 열만 빠진다.
_gen = pd.read_csv(VIZ / 'generations_fold0.csv')
assert {'pop_min_gcv', 'uniq_gcv', 'gcv_q05', 'gcv_q95'} <= set(_gen.columns),     'the generation log is missing the population statistics!!'
_top = pd.read_csv(VIZ / 'top_fold0.csv')
assert {'uid', 'p1', 'p2', 'op', 'expr'} <= set(_top.columns), 'the top-k log is missing lineage columns!!'
assert _top.expr.notna().all(), 'a top-k row has no expression!!'

print(f'\n*** Records complete — {_total / 1e6:.1f} MB in {VIZ.name}/ ***')
print(f'    model_fold{{f}}.json carries the returned genes and ridge weights; lineage_fold{{f}}.npz the parent-child edges')
print(f'    generations {len(_gen):,} rows · top-k {len(_top):,} rows (fold 0)')

# Kaggle's "Submit to Competition -> Notebook" list is the output tree in alphabetical order, so 30
# loose files under outputs/viz/ push submission.csv to the very bottom. One zip keeps the root short.
# 캐글 제출 목록은 출력 트리를 알파벳 순으로 늘어놓는다. outputs/viz/ 밑 느슨한 파일 30개가
# submission.csv를 맨 끝으로 밀어낸다. 점검 뒤 하나로 묶고, 이름을 's' 뒤로 정렬되게 둬 맨 위를 내준다.
_zip = ROOT_DIR / 'viz_records.zip'
with zipfile.ZipFile(_zip, 'w', zipfile.ZIP_DEFLATED) as _z:
    for _p in sorted(VIZ.iterdir()):
        _z.write(_p, _p.name)      # 디렉터리 없이 파일 이름만 넣는다 / file names only, no paths
shutil.rmtree(OUTPUT_DIR)

print(f'\n*** Packed -> {_zip.name}  {_zip.stat().st_size / 1e6:.1f} MB '
      f'(from {_total / 1e6:.1f} MB), download it to draw the figures ***')
print(f'    output root: submission.csv, {_zip.name} — submission.csv sorts first')
print(f'    출력 루트에는 submission.csv와 {_zip.name}뿐이고, submission.csv가 먼저 온다')

 fold   generations           top        census         genes         model       lineage
    0         95 KB        702 KB      1,646 KB        859 KB          1 KB     25,746 KB
    1         96 KB        861 KB      1,649 KB        894 KB          1 KB     27,970 KB
    2         95 KB        680 KB      1,645 KB        819 KB          1 KB     25,281 KB
    3         95 KB        890 KB      1,650 KB        926 KB          1 KB     27,439 KB
    4         96 KB        898 KB      1,649 KB        948 KB          1 KB     28,014 KB

*** Records complete — 155.3 MB in viz/ ***
    model_fold{f}.json carries the returned genes and ridge weights; lineage_fold{f}.npz the parent-child edges
    generations 301 rows · top-k 1,505 rows (fold 0)

*** Packed -> viz_records.zip  120.5 MB (from 155.3 MB), download it to draw the figures ***
    output root: submission.csv, viz_records.zip — submission.csv sorts first
    출력 루트에는 submission.csv와 viz_records.zip뿐이고, submission.csv가 먼저 온다


# 8. Submission / 제출

### Score gate / 점수 문턱

학습과 기록은 앞에서 이미 끝났고 로그와 `outputs/`도 남았다. 여기서 막히는 것은 제출뿐이다.<br>
<br>
fold valid는 반환 개체를 고르는 데 쓴 값이라 리더보드를 예측하지는 못한다. 그래도 기록과 나란히 놓으면
런이 망가졌는지는 안다. 시드와 환경이 다르면 수치는 움직이겠지만 많이 움직이면 안 된다.<br>
<br>
문턱은 둘이다.<br>
<br>
1. **기록 문턱.** fold 5개 valid 평균이 기록 0.1282보다 0.010 이상 나쁘면 막는다. 그 기록은 이 노트북을
같은 fold 표 위에서 본 예산(10,000 × 300세대)으로 끝까지 돌려 잰 값이다.<br>
2. **기준선 문턱.** 같은 fold의 릿지 90열을 이기지 못하면 막는다. 그 기준선은 4절에서 이 런이 직접 잰
값이다. 같은 행에서 재므로 분할 난이도가 달라져도 자로서 흔들리지 않는다. 기록 런은 평균 0.0161 앞섰다.<br>
<br>
0.010은 이렇게 잡았다. 같은 설정을 환경만 바꿔 돌린 두 런은 서로 +0.0002 안에 들어왔고, 반환 스냅샷을
10칸 늦게 골라도 +0.0043이다. 같은 평가 예산의 다른 배분(1000×300 · 3000×100 · 10000×30)도 기록의
±0.005 안에 들어온다. 반대쪽에는 릿지 90열이 +0.0161, 이전 세대 GP가 +0.030, 배선 확인용 작은 런이
+0.047 떨어져 있다. 문턱은 그 사이에 둔다. 0.005로 조이면 스냅샷 선택이 조금만 밀려도 정상 런이 막힌다.
QUICK 런에서는 수치만 찍고 막지 않는다.

In [22]:
# Check 3 of 4 — the score gate. Training and the records are already done; only the submission is gated.
# 검증 3/4 — 점수 문턱. 학습과 기록은 이미 끝났고, 여기서 막히는 것은 제출뿐이다.
# REF_VALID is the one number that comes from outside this notebook: an earlier run's score, which
# the data cannot tell you. The ridge baseline is measured here instead of quoted.
# REF_VALID만 이 노트북 밖에서 온 값이다. 지난 런의 점수라 데이터로는 알 수 없다. 릿지 기준선은
# 인용하지 않고 4절에서 직접 쟀다.
REF_VALID = 0.1282      # the record: this notebook's own full run, mean valid of the 5 fold models
                        # 기록 — 이 노트북을 본 예산으로 끝까지 돌린 런, fold 5개 valid 평균
REF_MARGIN = 0.0161     # how far that run beat ridge-90 on its own folds / 그 런이 릿지 90열을 이긴 폭
REF_TOL = 0.010         # the same setup in another environment landed +0.0002 away, and a snapshot
                        # picked 10 slots late +0.0043; ridge-90 sits +0.0161 away
                        # 환경만 바꾼 같은 설정은 +0.0002, 스냅샷을 10칸 늦게 골라도 +0.0043,
                        # 릿지 90열은 +0.0161 밖이다

gp = np.array([RES[f]['valid'] for f in range(5)])
ridge = np.array([RIDGE90[f] for f in range(5)])

print(f'{"fold":>5}{"GP valid":>10}{"ridge-90":>10}{"margin":>9}')
for f in range(5):
    print(f'{f:>5}{gp[f]:>10.4f}{ridge[f]:>10.4f}{ridge[f] - gp[f]:>+9.4f}')

gap = float(gp.mean() - REF_VALID)
margin = float(ridge.mean() - gp.mean())
print(f'\nmean GP valid {gp.mean():.4f} · record {REF_VALID:.4f} · gap {gap:+.4f} (tolerance {REF_TOL})')
print(f'margin over ridge-90 on these folds {margin:+.4f} · record margin {REF_MARGIN:+.4f}')

if QUICK:
    GATE_OK = True
    print('\nQUICK run — the gate is reported, not enforced.')
    print('    The submission below is a wiring artifact, not a result. To submit: set QUICK = False,')
    print('    then Save Version -> Save & Run All. Quick Save does not execute and leaves no output.')
    print('    QUICK 런의 제출물은 배선 확인용이다. 제출하려면 QUICK = False로 두고')
    print('    Save Version -> Save & Run All로 커밋해야 한다. Quick Save는 실행하지 않아 출력이 남지 않는다.')
else:
    assert gap < REF_TOL, \
        f'mean valid {gp.mean():.4f} is {gap:+.4f} off the record — significantly worse, submission blocked!!'
    assert margin > 0, \
        f'GP does not beat ridge-90 on its own folds (margin {margin:+.4f}) — submission blocked!!'
    GATE_OK = True
    print('\n*** Score gate passed — within tolerance of the record ***')

 fold  GP valid  ridge-90   margin
    0    0.1392    0.1380  -0.0013
    1    0.1488    0.1879  +0.0391
    2    0.1261    0.1513  +0.0252
    3    0.1207    0.1244  +0.0038
    4    0.1061    0.1198  +0.0137

mean GP valid 0.1282 · record 0.1282 · gap -0.0000 (tolerance 0.01)
margin over ridge-90 on these folds +0.0161 · record margin +0.0161

*** Score gate passed — within tolerance of the record ***


### Write submission.csv / 제출 파일 저장

In [23]:
# Check 4 of 4 — the submission contract. The ensemble is the mean of the five fold models in log space.
# 검증 4/4 — 제출 계약. 앙상블은 fold 모델 5개를 log 공간에서 평균한 것이다.
assert GATE_OK, 'the score gate has not passed — submission blocked!!'

P = np.array([RES[f]['pred_test'] for f in range(5)])
ens = P.mean(axis=0)

assert P.shape == (5, len(sample_ids)), f'prediction block {P.shape}!!'
assert np.isfinite(ens).all(), 'non-finite values in the ensemble!!'
# The range guard belongs here, on the averaged prediction, not on a single fold.
# 범위 검사는 fold 하나가 아니라 평균낸 예측에, 즉 여기에 건다.
assert (ens > Y_LO - 1.0).all() and (ens < Y_HI + 1.0).all(), \
    f'ensemble leaves the plausible price range [{ens.min():.2f}, {ens.max():.2f}]!!'

submission = pd.DataFrame({'Id': sample_ids, 'SalePrice': np.exp(ens)})

assert list(submission.columns) == ['Id', 'SalePrice'], 'submission columns != contract!!'
assert len(submission) == len(sample_ids), 'row count mismatch!!'
assert submission.notna().all().all(), 'NaN detected in submission!!'
assert submission.SalePrice.gt(0).all(), 'non-positive price!!'

submission.to_csv(SUB_PATH, index=False)
print(f'fold valid    : ' + ', '.join(f'{RES[f]["valid"]:.4f}' for f in range(5)))
print(f'spread between fold predictions (mean sd): {P.std(axis=0).mean():.4f}')
print(f'*** Submission written -> {SUB_PATH}  {len(submission):,} rows ***')
print(f'    SalePrice [{submission.SalePrice.min():,.0f}, {submission.SalePrice.max():,.0f}] '
      f'median {submission.SalePrice.median():,.0f}')

fold valid    : 0.1392, 0.1488, 0.1261, 0.1207, 0.1061
spread between fold predictions (mean sd): 0.0293
*** Submission written -> /kaggle/working/submission.csv  1,459 rows ***
    SalePrice [51,232, 533,860] median 158,073


In [24]:
submission.head()

,Id,SalePrice
0,1461,114126.332033
1,1462,164613.441672
2,1463,183783.537758
3,1464,203205.512694
4,1465,186361.357911


In [25]:
print('*** Done. ***')

*** Done. ***


# 끝.